# Linear Eigenvalue Analysis for Rotating-stratified Flows

## Dispersion data processing

In [3]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import glob
from matplotlib.colors import ListedColormap
from matplotlib.cm import get_cmap
import pandas as pd

def parse_eigenvalue_file(filename):
    """
    Parse eigenvalue files from MLegS output.
    
    Parameters:
    -----------
    filename : str
        Path to the eigenvalue file
        
    Returns:
    --------
    dict
        Dictionary containing the parsed data
    """
    # Extract parameters from filename
    # Update pattern to handle more number formats
    pattern1 = r"bsnsq_eig_bv_([0-9.-]+)_w_([.0-9+-]+)_m_(\d+)_k_([+-]?[0-9.-]+)_nr_([+-]?\d+)"
    match = re.search(pattern1, filename)
    
    if match:
        bv_freq = float(match.group(1))
        omega = float(match.group(2))
        m = int(match.group(3))
        k = float(match.group(4))
        nrchop = int(match.group(5))
    else:
        print(f"Couldn't parse parameters from {filename}")
        return None
    
    # Parse file content
    eigenvalues = []
    is_resolved = []
    indices = []
    
    with open(filename, 'r') as file:
        for line in file:
            # Match the eigenvalue format
            match = re.search(r'II\s*=\s*(\d+)\s*:\s*\(([^,]+),([^)]+)\)\s*-\s*([TF])', line)
            if match:
                index = int(match.group(1))
                real_part = float(match.group(2))
                imag_part = float(match.group(3))
                resolved = match.group(4) == 'T'
                
                indices.append(index)
                eigenvalues.append(complex(real_part, imag_part))
                is_resolved.append(resolved)
    
    return {
        'bv_freq': bv_freq,
        'omega': omega,
        'm': m,
        'k': k,
        'nrchop': nrchop,
        'indices': np.array(indices),
        'eigenvalues': np.array(eigenvalues),
        'is_resolved': np.array(is_resolved)
    }

def read_velocity_profile(data_dir, bv_freq, omega, m, k, index):
    """
    Read velocity profile data for a specific eigenvalue index.
    
    Parameters:
    -----------
    data_dir : str
        Base directory for data files
    bv_freq : float
        Brunt-Väisälä frequency
    omega : float
        Angular velocity value
    m : int
        Azimuthal wavenumber
    k : float
        Axial wavenumber
    index : int
        Eigenvalue index
        
    Returns:
    --------
    dict
        Dictionary with radial coordinates, velocity components, 
        and metadata
    """
    # Format the filename based on parameters
    bv_str = f"{bv_freq:.2f}".replace("0.", ".")
    w_str = f"{omega:.2f}".replace("0.", ".")
    k_str = f"{k:+.4f}".replace("0.", ".")
    
    vel_dir = f"{data_dir}/bsnsq_vel/bv_{bv_str}_w_{w_str}_m_{m}_k_{k_str}"
    vel_file = f"{vel_dir}/ind_{index}.output"
    
    try:
        # Try to read the velocity file
        data = []
        with open(vel_file, 'r') as file:
            for line in file:
                # Parse comma-separated values
                values = [float(val) for val in line.strip().split(',')]
                data.append(values)
        
        # Convert to numpy array
        data = np.array(data)
        
        # Create a structured output
        velocity_data = {
            'r': data[:, 0],
            'ur_real': data[:, 1],
            'ur_imag': data[:, 2],
            'uphi_real': data[:, 3],
            'uphi_imag': data[:, 4],
            'uz_real': data[:, 5],
            'uz_imag': data[:, 6],
            'b_real': data[:, 7],
            'b_imag': data[:, 8],
            'ur': data[:, 1] + 1j * data[:, 2],
            'uphi': data[:, 3] + 1j * data[:, 4],
            'uz': data[:, 5] + 1j * data[:, 6],
            'b': data[:, 7] + 1j * data[:, 8],
            'metadata': {
                'bv_freq': bv_freq,
                'omega': omega,
                'm': m,
                'k': k,
                'index': index
            }
        }
        
        return velocity_data
    
    except (FileNotFoundError, IOError) as e:
        print(f"Error reading velocity file {vel_file}: {e}")
        return None

# Function to plot velocity profiles for a specific eigenvalue
def plot_velocity_profiles(data_dir, bv_freq, omega, m, k, index, r_min=0, r_max=5):
    """
    Plot velocity profiles for a specific eigenvalue index.
    
    Parameters:
    -----------
    data_dir : str
        Base directory for data files
    bv_freq : float
        Brunt-Väisälä frequency
    omega : float
        Angular velocity value
    m : int
        Azimuthal wavenumber
    k : float
        Axial wavenumber
    index : int
        Eigenvalue index
    """
    vel_data = read_velocity_profile(data_dir, bv_freq, omega, m, k, index)
    
    if vel_data is None:
        print("No velocity data available.")
        return
    
    # Create a figure with 3 subplots for ur, uphi, and uz
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    # Get the radial coordinate
    r = vel_data['r']
    
    # Calculate magnitudes
    ur_mag = np.abs(vel_data['ur'])
    uphi_mag = np.abs(vel_data['uphi'])
    uz_mag = np.abs(vel_data['uz'])
    
    # Plot radial velocity
    ax1.plot(r, vel_data['ur_real'], 'b-', label='Real')
    ax1.plot(r, vel_data['ur_imag'], 'r-', label='Imaginary')
    ax1.plot(r, ur_mag, 'k--', label='Magnitude')
    ax1.set_xlabel('r')
    ax1.set_ylabel('ur')
    ax1.set_title('Radial Velocity')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_xlim(r_min, r_max)
    
    # Plot azimuthal velocity
    ax2.plot(r, vel_data['uphi_real'], 'b-', label='Real')
    ax2.plot(r, vel_data['uphi_imag'], 'r-', label='Imaginary')
    ax2.plot(r, uphi_mag, 'k--', label='Magnitude')
    ax2.set_xlabel('r')
    ax2.set_ylabel('uφ')
    ax2.set_title('Azimuthal Velocity')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_xlim(r_min, r_max)
    
    # Plot axial velocity
    ax3.plot(r, vel_data['uz_real'], 'b-', label='Real')
    ax3.plot(r, vel_data['uz_imag'], 'r-', label='Imaginary')
    ax3.plot(r, uz_mag, 'k--', label='Magnitude')
    ax3.set_xlabel('r')
    ax3.set_ylabel('uz')
    ax3.set_title('Axial Velocity')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    ax3.set_xlim(r_min, r_max)
    
    # Add a main title
    # meta = vel_data['metadata']
    # fig.suptitle(f'Velocity Profiles for BV={meta["bv_freq"]}, ω={meta["omega"]}, m={meta["m"]}, k={meta["k"]}, Index={meta["index"]}', 
    #             fontsize=14, y=1.02)
    
    plt.tight_layout()
    plt.show()

    return vel_data

# Example usage:
# velocity_data = read_velocity_profile("/Users/jinge/Projects/MLegS-dev/MLegS/data", 1.11, 0.00, 1, 0.0000, 1)
# plot_velocity_profiles("/Users/jinge/Projects/MLegS-dev/MLegS/data", 1.11, 0.00, 1, 0.0000, 1)

In [2]:
from matplotlib.lines import Line2D

# Find all eigenvalue files
# data_dir = "/Users/jinge/Projects/MLegS-dev/MLegS/data"
data_dir = "/home/x-jinge/FOLDER_SCRATCH/Bousinessq/MLegS/data"
files = glob.glob(os.path.join(data_dir, 'bsnsq_eig_bv_*.output'))

# Process all files
all_data = []
for file in files:
    data = parse_eigenvalue_file(file)
    if data:
        all_data.append(data)

print(f"Processed {len(all_data)} files.")

Processed 0 files.


## Dispersion plot w/ velocity profiles

In [ ]:
from ipywidgets import VBox, Output, Dropdown, Layout
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import matplotlib.pyplot as plt

def create_plotly_velocity_profiles(vel_data=None, r_min=0, r_max=5):
    """
    Create Plotly figure for velocity profiles and density.
    
    Parameters:
    -----------
    vel_data : dict or None
        Dictionary with velocity data, if None creates empty plot
    r_min : float
        Minimum r value for plot range
    r_max : float
        Maximum r value for plot range
        
    Returns:
    --------
    fig : go.FigureWidget
        Plotly figure widget that can be updated later
    """
    # Create a figure with 4 subplots for ur, uphi, uz, and density
    fig = make_subplots(
        rows=1, cols=4,
        subplot_titles=("Radial Velocity", "Azimuthal Velocity", "Axial Velocity", "Density"),
        shared_xaxes=True,
        horizontal_spacing=0.05
    )
    
    if vel_data is None:
        # Create empty plots with placeholders for all 4 subplots
        for col in range(1, 5):  # columns 1, 2, 3, 4
            fig.add_trace(go.Scatter(x=[], y=[], name='Real', line=dict(color='blue'), 
                                   showlegend=(col==1)), row=1, col=col)
            fig.add_trace(go.Scatter(x=[], y=[], name='Imaginary', line=dict(color='red'), 
                                   showlegend=(col==1)), row=1, col=col)
        
        title_text = "Velocity Profiles and Density (No Data Selected)"
    else:
        # Extract data for plotting
        r = vel_data['r']
        
        # Filter data to r_min/r_max range
        mask = (r >= r_min) & (r <= r_max)
        r = r[mask]
        ur_real = vel_data['ur_real'][mask]
        ur_imag = vel_data['ur_imag'][mask]
        uphi_real = vel_data['uphi_real'][mask]
        uphi_imag = vel_data['uphi_imag'][mask]
        uz_real = vel_data['uz_real'][mask]
        uz_imag = vel_data['uz_imag'][mask]
        b_real = vel_data['b_real'][mask]
        b_imag = vel_data['b_imag'][mask]
        
        # Add traces for radial velocity
        fig.add_trace(go.Scatter(x=r, y=ur_real, name='Real', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=r, y=ur_imag, name='Imaginary', line=dict(color='red')), row=1, col=1)
        
        # Add traces for azimuthal velocity
        fig.add_trace(go.Scatter(x=r, y=uphi_real, name='Real', line=dict(color='blue'), showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=r, y=uphi_imag, name='Imaginary', line=dict(color='red'), showlegend=False), row=1, col=2)
        
        # Add traces for axial velocity
        fig.add_trace(go.Scatter(x=r, y=uz_real, name='Real', line=dict(color='blue'), showlegend=False), row=1, col=3)
        fig.add_trace(go.Scatter(x=r, y=uz_imag, name='Imaginary', line=dict(color='red'), showlegend=False), row=1, col=3)
        
        # Add traces for density
        fig.add_trace(go.Scatter(x=r, y=b_real, name='Real', line=dict(color='blue'), showlegend=False), row=1, col=4)
        fig.add_trace(go.Scatter(x=r, y=b_imag, name='Imaginary', line=dict(color='red'), showlegend=False), row=1, col=4)
        
        # Create a title with metadata
        meta = vel_data['metadata']
        title_text = f"Velocity Profiles and Density (BV={meta['bv_freq']}, ω={meta['omega']}, m={meta['m']}, k={meta['k']}, Index={meta['index']})"
    
    # Update layout and axis labels
    fig.update_layout(
        height=500,
        width=1600,
        title_text=title_text,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=50, r=50, t=80, b=50),
        hovermode="closest"
    )
    
    # Update x-axis labels for all 4 subplots
    for col in range(1, 5):
        fig.update_xaxes(title_text="r", range=[r_min, r_max], row=1, col=col)
    
    # Update y-axis labels
    fig.update_yaxes(title_text="ur", row=1, col=1)
    fig.update_yaxes(title_text="uφ", row=1, col=2)
    fig.update_yaxes(title_text="uz", row=1, col=3)
    fig.update_yaxes(title_text="ρ", row=1, col=4)
    
    # Add grid lines
    fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.1)')
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.1)')
    
    return go.FigureWidget(fig)

def update_plotly_velocity_profiles(fig_widget, data_dir, bv_freq, omega, m, k, index, r_min=0, r_max=5):
    """
    Update an existing Plotly figure widget with new velocity profile and density data.
    
    Parameters:
    -----------
    fig_widget : go.FigureWidget
        Existing figure widget to update
    data_dir : str
        Base directory for data files
    bv_freq, omega, m, k, index : parameters for the eigenvalue
    r_min, r_max : float
        Plot range limits
        
    Returns:
    --------
    success : bool
        True if update was successful, False otherwise
    """
    # Read the velocity profile data
    vel_data = read_velocity_profile(data_dir, bv_freq, omega, m, k, index)
    
    if vel_data is None:
        # Update title to show error
        fig_widget.update_layout(title_text="Velocity Profile and Density Data Not Available")
        # Clear all traces
        for i in range(len(fig_widget.data)):
            fig_widget.data[i].x = []
            fig_widget.data[i].y = []
        return False
    
    # Extract data for plotting
    r = vel_data['r']
    
    # Filter data to r_min/r_max range
    mask = (r >= r_min) & (r <= r_max)
    r_filtered = r[mask]
    
    # Extract filtered data
    ur_real = vel_data['ur_real'][mask]
    ur_imag = vel_data['ur_imag'][mask]
    
    uphi_real = vel_data['uphi_real'][mask]
    uphi_imag = vel_data['uphi_imag'][mask]
    
    uz_real = vel_data['uz_real'][mask]
    uz_imag = vel_data['uz_imag'][mask]
    
    b_real = vel_data['b_real'][mask]
    b_imag = vel_data['b_imag'][mask]
    
    # Calculate individual subplot y-axis ranges
    ur_max = max(abs(ur_real.max()), abs(ur_real.min()), abs(ur_imag.max()), abs(ur_imag.min()))
    uphi_max = max(abs(uphi_real.max()), abs(uphi_real.min()), abs(uphi_imag.max()), abs(uphi_imag.min()))
    uz_max = max(abs(uz_real.max()), abs(uz_real.min()), abs(uz_imag.max()), abs(uz_imag.min()))
    b_max = max(abs(b_real.max()), abs(b_real.min()), abs(b_imag.max()), abs(b_imag.min()))
    
    # Update the figure with the new data using batch_update for efficiency
    with fig_widget.batch_update():
        # Radial velocity (traces 0, 1)
        fig_widget.data[0].x = r_filtered
        fig_widget.data[0].y = ur_real
        fig_widget.data[1].x = r_filtered
        fig_widget.data[1].y = ur_imag
        
        # Azimuthal velocity (traces 2, 3)
        fig_widget.data[2].x = r_filtered
        fig_widget.data[2].y = uphi_real
        fig_widget.data[3].x = r_filtered
        fig_widget.data[3].y = uphi_imag
        
        # Axial velocity (traces 4, 5)
        fig_widget.data[4].x = r_filtered
        fig_widget.data[4].y = uz_real
        fig_widget.data[5].x = r_filtered
        fig_widget.data[5].y = uz_imag
        
        # Density (traces 6, 7)
        fig_widget.data[6].x = r_filtered
        fig_widget.data[6].y = b_real
        fig_widget.data[7].x = r_filtered
        fig_widget.data[7].y = b_imag
        
        # Update title with metadata
        meta = vel_data['metadata']
        title_text = f"Velocity Profiles and Density (BV={meta['bv_freq']}, ω={meta['omega']}, m={meta['m']}, k={meta['k']}, Index={meta['index']})"
        fig_widget.update_layout(title_text=title_text)
        
        # Update axis ranges individually for each subplot with some padding
        ur_margin = ur_max * 0.1
        uphi_margin = uphi_max * 0.1
        uz_margin = uz_max * 0.1
        b_margin = b_max * 0.1
        
        fig_widget.update_yaxes(range=[-ur_max-ur_margin, ur_max+ur_margin], row=1, col=1)
        fig_widget.update_yaxes(range=[-uphi_max-uphi_margin, uphi_max+uphi_margin], row=1, col=2)
        fig_widget.update_yaxes(range=[-uz_max-uz_margin, uz_max+uz_margin], row=1, col=3)
        fig_widget.update_yaxes(range=[-b_max-b_margin, b_max+b_margin], row=1, col=4)
    
    return True

def plot_dispersion_interactive(data_list, m_value=None, omega=None, bv_freq=None, eig_min=0, eig_max=None, r_min=0, r_max=5, kFr_min=0, kFr_max=5):
    # Filter data by bv_freq if specified
    if bv_freq is not None:
        data_list = [d for d in data_list if d['bv_freq'] == bv_freq]
    # Filter data by omega if specified
    if omega is not None:
        data_list = [d for d in data_list if d['omega'] == omega]
    
    # Get unique m values for the dropdown
    m_values = sorted(list(set(d['m'] for d in data_list)))
    
    if m_value is None and m_values:
        m_value = m_values[0]
    
    # Prepare output widgets
    out_disp = Output()
    out_vel = Output()
    
    # Create dropdown for m values
    m_dropdown = Dropdown(
        options=[(f'm={m}', m) for m in m_values],
        value=m_value,
        description='Mode:',
        layout=Layout(width='200px')
    )
    
    # Pre-process data by m value
    data_by_m = {}
    for m in m_values:
        group = [d for d in data_list if d['m'] == m]
        group.sort(key=lambda x: x['k'])
        if group and group[0]['omega'] == 0:
            group = [d for d in group if d['k'] >= 0]
        data_by_m[m] = group
        
    # Pre-process points data
    points_by_m = {}
    for m, group in data_by_m.items():
        if not group:
            continue
            
        if group[0]['bv_freq'] == 0:
            Fr = 1
        else:
            Fr = 1 / group[0]['bv_freq']
        points_data = []
        for d in group:
            eigenvalues = d['eigenvalues']
            is_resolved = d['is_resolved']
            k_Fr = d['k'] * Fr
            indices = d['indices']
            for i, (eig, idx, resolved) in enumerate(zip(eigenvalues, indices, is_resolved)):
                point_data = {
                    'real': eig.real,
                    'imag': eig.imag,
                    'k_Fr': k_Fr,
                    'bv_freq': d['bv_freq'],
                    'omega': d['omega'],
                    'm': d['m'],
                    'k': d['k'],
                    'index': int(idx),
                    'resolved': resolved
                }
                points_data.append(point_data)
        points_by_m[m] = points_data

    # Compute eig_max if not provided
    if eig_max is None:
        all_real_values = []
        for m in points_by_m:
            if points_by_m[m]:  # Check if the list is not empty
                all_real_values.extend([pt['real'] for pt in points_by_m[m]])
        eig_max = max(all_real_values) if all_real_values else 1

    # Function to create plot for a given m value
    def create_plot_for_m(m):
        if m not in points_by_m or not points_by_m[m]:
            with out_disp:
                out_disp.clear_output(wait=True)
                print(f"No data found for m={m}")
            return None, None
        
        # Get all points for this mode number
        all_points = points_by_m[m]
        
        # Determine k_min and k_max based on parameters or data range
        all_k_Fr = [pt['k_Fr'] for pt in all_points]
        k_min = kFr_min if kFr_min is not None else (min(all_k_Fr) if all_k_Fr else 0)
        k_max = kFr_max if kFr_max is not None else (max(all_k_Fr) if all_k_Fr else 1)
        
        # Filter points by k range before any other processing
        filtered_points = [pt for pt in all_points if k_min <= pt['k_Fr'] <= k_max]
        
        # Split into resolved and unresolved points
        resolved_points = [pt for pt in filtered_points if pt['resolved']]
        unresolved_points = [pt for pt in filtered_points if not pt['resolved']]
        
        # Extract data for plotting only from filtered points
        k_Fr_resolved = [pt['k_Fr'] for pt in resolved_points]
        real_resolved = [pt['real'] for pt in resolved_points]
        imag_resolved = [pt['imag'] for pt in resolved_points]
        
        k_Fr_unresolved = [pt['k_Fr'] for pt in unresolved_points]
        real_unresolved = [pt['real'] for pt in unresolved_points]
        imag_unresolved = [pt['imag'] for pt in unresolved_points]
        
        # Create subplots
        fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=("Real Part", "Imaginary Part"),
                        horizontal_spacing=0.1)
        
        # Add traces for real part (unresolved)
        fig.add_trace(
            go.Scatter(
                x=k_Fr_unresolved, y=real_unresolved,
                mode='markers', name='Unresolved',
                marker=dict(size=3, color='blue', opacity=1),
                customdata=[{
                    'k': pt['k'],
                    'index': pt['index'],
                    'real': pt['real'],
                    'imag': pt['imag'],
                    'm': pt['m'],
                    'bv_freq': pt['bv_freq'],
                    'omega': pt['omega'],
                    'point_idx': all_points.index(pt)  # Find original index for reference
                } for pt in unresolved_points],
                hovertemplate='k*Fr: %{x:.3f}<br>Re(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=1
        )

        # Add traces for real part (resolved)
        fig.add_trace(
            go.Scatter(
                x=k_Fr_resolved, y=real_resolved,
                mode='markers', name='Resolved',
                marker=dict(size=5, color='red', opacity=1),
                customdata=[{
                    'k': pt['k'],
                    'index': pt['index'],
                    'real': pt['real'],
                    'imag': pt['imag'],
                    'm': pt['m'],
                    'bv_freq': pt['bv_freq'],
                    'omega': pt['omega'],
                    'point_idx': all_points.index(pt)
                } for pt in resolved_points],
                hovertemplate='k*Fr: %{x:.3f}<br>Re(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=1
        )
        
        # Add traces for imaginary part (unresolved)
        fig.add_trace(
            go.Scatter(
                x=k_Fr_unresolved, y=imag_unresolved,
                mode='markers', name='Unresolved',
                marker=dict(size=3, color='blue', opacity=1),
                customdata=[{
                    'k': pt['k'],
                    'index': pt['index'],
                    'real': pt['real'],
                    'imag': pt['imag'],
                    'm': pt['m'],
                    'bv_freq': pt['bv_freq'],
                    'omega': pt['omega'],
                    'point_idx': all_points.index(pt)
                } for pt in unresolved_points],
                hovertemplate='k*Fr: %{x:.3f}<br>Im(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=2
        )

        # Add traces for imaginary part (resolved)
        fig.add_trace(
            go.Scatter(
                x=k_Fr_resolved, y=imag_resolved,
                mode='markers', name='Resolved',
                marker=dict(size=5, color='red', opacity=1),
                customdata=[{
                    'k': pt['k'],
                    'index': pt['index'],
                    'real': pt['real'],
                    'imag': pt['imag'],
                    'm': pt['m'],
                    'bv_freq': pt['bv_freq'],
                    'omega': pt['omega'],
                    'point_idx': all_points.index(pt)
                } for pt in resolved_points],
                hovertemplate='k*Fr: %{x:.3f}<br>Im(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=2
        )
        
        # Pre-create highlight traces (hidden)
        fig.add_scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color='rgba(0,0,0,0)', 
                    line=dict(color='green', width=3)),
            showlegend=False, 
            name='Highlight Real',
            hoverinfo='none',
            row=1, col=1
        )
        
        fig.add_scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color='rgba(0,0,0,0)', 
                    line=dict(color='green', width=3)),
            showlegend=False, 
            name='Highlight Imag',
            hoverinfo='none',
            row=1, col=2
        )

        # Update layout
        bv = data_by_m[m][0]['bv_freq']
        w = data_by_m[m][0]['omega']
        fig.update_layout(
            title_text=f'Dispersion Relations (BV={bv}, Ω̄={w}, m={m})',
            height=800,
            width=1600,
            clickmode='event',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        # Calculate y-axis range based on filtered data
        all_filtered_real = real_resolved + real_unresolved

        if all_filtered_real:
            local_eig_max = min(max(all_filtered_real), eig_max)
        else:
            local_eig_max = eig_max
        
        # Set axis properties
        fig.update_xaxes(title_text='k*Fr', range=[k_min, k_max], row=1, col=1)
        fig.update_xaxes(title_text='k*Fr', range=[k_min, k_max], row=1, col=2)
        fig.update_yaxes(title_text='Re(σ)', range=[eig_min, local_eig_max], row=1, col=1)
        fig.update_yaxes(title_text='Im(σ)', row=1, col=2)
        
        # Add some padding to x-range for better visualization
        padding = (k_max - k_min) * 0.1
        fig.update_xaxes(range=[k_min - padding, k_max + padding], row=1, col=1)
        fig.update_xaxes(range=[k_min - padding, k_max + padding], row=1, col=2)
        fig.update_yaxes(range=[eig_min, local_eig_max], row=1, col=1)
        
        return fig, all_points
    
    # Global variables to track current state
    current_fig_widget = None
    current_points_data = None
    selected_point_info = None
    
    # Function to handle point clicks and create highlights
    def handle_click(trace, points, selector):
        nonlocal selected_point_info
        
        if not points.point_inds or current_fig_widget is None:
            return
            
        pt_idx = points.point_inds[0]
        
        # Get the data for the clicked point
        if trace.customdata and len(trace.customdata) > pt_idx:
            custom_data = trace.customdata[pt_idx]
            point_idx = custom_data['point_idx']
            selected_pt = current_points_data[point_idx]
            selected_point_info = selected_pt

            # Update the highlight marker positions
            with current_fig_widget.batch_update():
                # Get the indices of the highlight traces (should be the last two traces)
                highlight_real_idx = 4  # Index of the real highlight trace
                highlight_imag_idx = 5  # Index of the imaginary highlight trace
                
                # Update highlight in real part plot (subplot 1,1)
                current_fig_widget.data[highlight_real_idx].x = [selected_pt['k_Fr']]
                current_fig_widget.data[highlight_real_idx].y = [selected_pt['real']]
                
                # Update highlight in imaginary part plot (subplot 1,2)
                current_fig_widget.data[highlight_imag_idx].x = [selected_pt['k_Fr']]
                current_fig_widget.data[highlight_imag_idx].y = [selected_pt['imag']]
            
            # Update velocity profile panel
            with out_vel:
                out_vel.clear_output(wait=True)
                try:
                    success = update_plotly_velocity_profiles(
                        vel_fig_widget, 
                        data_dir, 
                        selected_pt['bv_freq'], 
                        selected_pt['omega'], 
                        selected_pt['m'], 
                        selected_pt['k'], 
                        selected_pt['index'],
                        r_min=r_min,
                        r_max=r_max
                    )
                    
                    if not success:
                        print(f"No velocity data available for m={selected_pt['m']}, k={selected_pt['k']}, index={selected_pt['index']}")
                        
                    display(vel_fig_widget)
                except Exception as e:
                    print(f"Error updating velocity profile: {e}")
    
    # Update function when dropdown changes
    def on_m_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            m = change['new']
            with out_disp:
                out_disp.clear_output(wait=True)
                nonlocal current_fig_widget, current_points_data
                
                fig, points_data = create_plot_for_m(m)
                if fig is None:
                    return
                
                current_points_data = points_data
                current_fig_widget = go.FigureWidget(fig)
                
                # Set up click callbacks for all data traces
                for i in range(4):  # We have 4 data traces (resolved/unresolved for real/imag)
                    if i < len(current_fig_widget.data):
                        current_fig_widget.data[i].on_click(handle_click)
                
                display(current_fig_widget)
                
                # Clear velocity profile
                with out_vel:
                    out_vel.clear_output(wait=True)
                    print("Click on a point to see velocity profile")
    
    # Initialize the plot
    fig, points_data = create_plot_for_m(m_value)
    if fig is None:
        return VBox([m_dropdown, out_disp, out_vel])
    
    current_points_data = points_data
    current_fig_widget = go.FigureWidget(fig)
    vel_fig_widget = create_plotly_velocity_profiles(r_min=r_min, r_max=r_max)

    # Set up the initial callbacks for all data traces
    for i in range(4):  # We have 4 data traces
        if i < len(current_fig_widget.data):
            current_fig_widget.data[i].on_click(handle_click)
    
    # Connect the dropdown callback
    m_dropdown.observe(on_m_change, names='value')
    
    # Initial display
    with out_disp:
        display(current_fig_widget)
    
    with out_vel:
        print("Click on a point to see velocity profile")
    
    # Return the full widget layout
    return VBox([
        m_dropdown,
        out_disp,
        out_vel
    ])


In [ ]:
plot_dispersion_interactive(all_data, m_value=0, omega=-0.6, bv_freq=0.0, kFr_max=5)

In [ ]:
plot_dispersion_interactive(all_data, m_value=0, omega=-0.6, bv_freq=1.11)

In [ ]:
plot_dispersion_interactive(all_data, m_value=0, omega=-1.2)

## Mode tracking

In [ ]:
def track_modes(data_list, m, bv_freq, omega, k_values=None, sim_threshold=0.8, eig_weight=0.4, vel_weight=0.6, 
                start_with_resolved_only=True, allow_unresolved_continuation=True):
    """
    Track eigenvalue modes across different k values.
    
    Parameters:
    -----------
    data_list : list
        List of parsed eigenvalue data dictionaries
    m : int
        Azimuthal wavenumber to track
    bv_freq : float
        Brunt-Väisälä frequency
    omega : float
        Background rotation rate
    k_values : list, optional
        Specific k values to track (defaults to all k values for the given m, bv_freq, omega)
    sim_threshold : float
        Similarity threshold for matching modes (0-1)
    eig_weight : float
        Weight for eigenvalue similarity in the matching score (0-1)
    vel_weight : float
        Weight for velocity profile similarity in the matching score (0-1)
    start_with_resolved_only : bool
        If True, only track modes that start as resolved at the first k value
    allow_unresolved_continuation : bool
        If True, allow resolved modes to continue into unresolved modes
        
    Returns:
    --------
    dict
        Dictionary with tracked modes information
    """
    # Filter data by m, bv_freq, omega
    filtered_data = [d for d in data_list if d['m'] == m and 
                     abs(d['bv_freq'] - bv_freq) < 1e-6 and 
                     abs(d['omega'] - omega) < 1e-6]
    
    if not filtered_data:
        print(f"No data found for m={m}, bv_freq={bv_freq}, omega={omega}")
        return None
    
    # Sort by k value
    filtered_data.sort(key=lambda d: d['k'])
    
    # If k_values is not specified, use all available k values
    if k_values is None:
        k_values = [d['k'] for d in filtered_data]
    else:
        # Filter data further to only include the specified k values
        filtered_data = [d for d in filtered_data if d['k'] in k_values]
        k_values = sorted([d['k'] for d in filtered_data])
    
    if len(k_values) < 2:
        print("At least two k values are needed for mode tracking")
        return None
    
    print(f"Tracking modes for m={m}, bv_freq={bv_freq}, omega={omega} across {len(k_values)} k values")
    
    # Initialize tracked modes dictionary
    tracked_modes = {
        'parameters': {
            'm': m,
            'bv_freq': bv_freq,
            'omega': omega,
        },
        'k_values': k_values,
        'modes': []
    }
    
    # Get data for the first k value
    first_k_data = next(d for d in filtered_data if d['k'] == k_values[0])
    first_k_eigvals = first_k_data['eigenvalues']
    first_k_indices = first_k_data['indices']
    first_k_resolved = first_k_data['is_resolved']
    
    # Initialize modes with data from the first k value
    # If start_with_resolved_only is True, only include resolved modes
    for i, (eig, idx, resolved) in enumerate(zip(first_k_eigvals, first_k_indices, first_k_resolved)):
        if not start_with_resolved_only or resolved:
            # Initialize a new mode with the first k value
            new_mode = {
                'k_data': [{
                    'k': k_values[0],
                    'eigenvalue': eig,
                    'index': int(idx),
                    'data_index': i,
                    'is_resolved': resolved
                }],
                'mode_id': len(tracked_modes['modes']),
                'k_coverage': [k_values[0]]
            }
            tracked_modes['modes'].append(new_mode)
    
    print(f"Starting with {len(tracked_modes['modes'])} " + 
          ("resolved " if start_with_resolved_only else "") + 
          f"modes at k={k_values[0]}")
    
    # Function to compute similarity between eigenmodes
    def compute_similarity(k1, idx1, k2, idx2):
        """Compute similarity between two eigenmodes"""
        # Get velocity profiles
        vel1 = read_velocity_profile(data_dir, bv_freq, omega, m, k1, idx1)
        vel2 = read_velocity_profile(data_dir, bv_freq, omega, m, k2, idx2)
        
        if vel1 is None or vel2 is None:
            return 0.0
        
        # Find eigenvalues for the modes
        k1_data = next(d for d in filtered_data if abs(d['k'] - k1) < 1e-6)
        k2_data = next(d for d in filtered_data if abs(d['k'] - k2) < 1e-6)
        
        # Get eigenvalue indices
        idx1_pos = np.where(k1_data['indices'] == idx1)[0][0]
        idx2_pos = np.where(k2_data['indices'] == idx2)[0][0]
        
        eig1 = k1_data['eigenvalues'][idx1_pos]
        eig2 = k2_data['eigenvalues'][idx2_pos]
        
        # Compute eigenvalue similarity - inversely proportional to distance
        eig_dist = abs(eig1 - eig2)
        eig_sim = 1.0 / (1.0 + eig_dist)
        
        # Compute velocity profile similarity
        # Interpolate to common r-grid if needed
        r1, r2 = vel1['r'], vel2['r']
        
        # Use common r range
        r_min = max(r1.min(), r2.min())
        r_max = min(r1.max(), r2.max())
        
        # Filter to common range
        mask1 = (r1 >= r_min) & (r1 <= r_max)
        mask2 = (r2 >= r_min) & (r2 <= r_max)
        
        r1_filtered = r1[mask1]
        ur1_real = vel1['ur_real'][mask1]
        uphi1_real = vel1['uphi_real'][mask1]
        uz1_real = vel1['uz_real'][mask1]
        b1_real = vel1['b_real'][mask1]
        
        # Interpolate vel2 to r1_filtered grid
        from scipy.interpolate import interp1d
        
        ur2_real_interp = interp1d(r2[mask2], vel2['ur_real'][mask2], bounds_error=False, fill_value=0)
        uphi2_real_interp = interp1d(r2[mask2], vel2['uphi_real'][mask2], bounds_error=False, fill_value=0)
        uz2_real_interp = interp1d(r2[mask2], vel2['uz_real'][mask2], bounds_error=False, fill_value=0)
        b2_real_interp = interp1d(r2[mask2], vel2['b_real'][mask2], bounds_error=False, fill_value=0)
        
        ur2_real = ur2_real_interp(r1_filtered)
        uphi2_real = uphi2_real_interp(r1_filtered)
        uz2_real = uz2_real_interp(r1_filtered)
        b2_real = b2_real_interp(r1_filtered)
        
        # Compute correlation coefficients for each component
        ur_corr = np.corrcoef(ur1_real, ur2_real)[0, 1]
        uphi_corr = np.corrcoef(uphi1_real, uphi2_real)[0, 1]
        uz_corr = np.corrcoef(uz1_real, uz2_real)[0, 1]
        b_corr = np.corrcoef(b1_real, b2_real)[0, 1]
        
        # Handle NaN values (can happen if arrays are constant)
        ur_corr = 0 if np.isnan(ur_corr) else ur_corr
        uphi_corr = 0 if np.isnan(uphi_corr) else uphi_corr
        uz_corr = 0 if np.isnan(uz_corr) else uz_corr
        b_corr = 0 if np.isnan(b_corr) else b_corr
        
        # Average correlation across all components
        vel_sim = (abs(ur_corr) + abs(uphi_corr) + abs(uz_corr) + abs(b_corr)) / 4.0
        
        # Compute weighted similarity
        total_sim = eig_weight * eig_sim + vel_weight * vel_sim
        
        return total_sim
    
    # Track modes across k values
    for k_idx in range(1, len(k_values)):
        curr_k = k_values[k_idx]
        prev_k = k_values[k_idx - 1]
        
        print(f"Tracking from k={prev_k} to k={curr_k}...")
        
        # Get data for the current k value
        curr_k_data = next(d for d in filtered_data if d['k'] == curr_k)
        curr_k_eigvals = curr_k_data['eigenvalues']
        curr_k_indices = curr_k_data['indices']
        curr_k_resolved = curr_k_data['is_resolved']
        
        # Create a list of available modes for the current k value
        # Include all modes regardless of resolved status if we're allowing unresolved continuation
        available_modes = [(i, idx, eig, resolved) for i, (eig, idx, resolved) 
                         in enumerate(zip(curr_k_eigvals, curr_k_indices, curr_k_resolved))]
        
        # List to keep track of which modes have been matched
        matched_curr_modes = [False] * len(available_modes)
        
        # For each previously tracked mode, find the best match
        for mode in tracked_modes['modes']:
            if prev_k in mode['k_coverage']:
                # Get the last entry for this mode
                last_entry = mode['k_data'][-1]
                last_k = last_entry['k']
                last_idx = last_entry['index']
                last_resolved = last_entry['is_resolved']
                
                # Compute similarity with all available modes at current k
                similarities = []
                for i, (curr_i, curr_idx, curr_eig, curr_resolved) in enumerate(available_modes):
                    if not matched_curr_modes[i]:  # Only consider unmatched modes
                        # If we don't allow unresolved continuation and this mode is unresolved, skip it
                        if not allow_unresolved_continuation and not curr_resolved:
                            continue
                            
                        sim = compute_similarity(last_k, last_idx, curr_k, curr_idx, r_range=(0, 20))
                        similarities.append((i, sim))
                
                # Find the best match above the threshold
                if similarities:
                    best_match = max(similarities, key=lambda x: x[1])
                    best_idx, best_sim = best_match
                    
                    if best_sim >= sim_threshold:
                        # Match found, update the mode
                        curr_i, curr_idx, curr_eig, curr_resolved = available_modes[best_idx]
                        mode['k_data'].append({
                            'k': curr_k,
                            'eigenvalue': curr_eig,
                            'index': int(curr_idx),
                            'data_index': curr_i,
                            'is_resolved': curr_resolved
                        })
                        mode['k_coverage'].append(curr_k)
                        matched_curr_modes[best_idx] = True
                        
                        # Print status including resolved/unresolved transition info
                        status = f"Mode {mode['mode_id']} continued with similarity {best_sim:.3f}"
                        if last_resolved and not curr_resolved:
                            status += " (resolved → unresolved)"
                        elif not last_resolved and curr_resolved:
                            status += " (unresolved → resolved)"
                        print(status)
        
        # Check for any new resolved modes that appeared at this k value
        # Only add new modes if they're resolved (we're already tracking the important ones)
        for i, matched in enumerate(matched_curr_modes):
            if not matched:
                curr_i, curr_idx, curr_eig, curr_resolved = available_modes[i]
                
                # Only add new modes if they're resolved
                if curr_resolved:
                    # Create a new mode
                    new_mode = {
                        'k_data': [{
                            'k': curr_k,
                            'eigenvalue': curr_eig,
                            'index': int(curr_idx),
                            'data_index': curr_i,
                            'is_resolved': curr_resolved
                        }],
                        'mode_id': len(tracked_modes['modes']),
                        'k_coverage': [curr_k]
                    }
                    tracked_modes['modes'].append(new_mode)
                    print(f"New resolved mode appeared with ID {new_mode['mode_id']} at k={curr_k}")
    
    # Compute continuity statistics
    continuity_stats = {}
    for k_value in k_values:
        continuity_stats[k_value] = {
            'total_modes': len([m for m in tracked_modes['modes'] if k_value in m['k_coverage']]),
            'resolved_modes': len([m for m in tracked_modes['modes'] 
                                 if k_value in m['k_coverage'] and 
                                 next(entry for entry in m['k_data'] if entry['k'] == k_value)['is_resolved']]),
            'unresolved_modes': len([m for m in tracked_modes['modes'] 
                                   if k_value in m['k_coverage'] and 
                                   not next(entry for entry in m['k_data'] if entry['k'] == k_value)['is_resolved']]),
            'continuing_modes': 0,
            'new_modes': 0,
            'disappearing_modes': 0
        }
    
    for mode in tracked_modes['modes']:
        k_coverage = mode['k_coverage']
        for i, k in enumerate(k_values):
            if k in k_coverage:
                if i > 0 and k_values[i-1] in k_coverage:
                    continuity_stats[k]['continuing_modes'] += 1
                else:
                    continuity_stats[k]['new_modes'] += 1
                if i < len(k_values)-1 and k_values[i+1] not in k_coverage:
                    continuity_stats[k]['disappearing_modes'] += 1
    
    tracked_modes['continuity_stats'] = continuity_stats
    
    # Calculate resolved/unresolved transition statistics
    transitions = {
        'resolved_to_unresolved': 0,
        'unresolved_to_resolved': 0,
        'stable_resolved': 0,
        'stable_unresolved': 0
    }
    
    for mode in tracked_modes['modes']:
        if len(mode['k_data']) >= 2:
            for i in range(1, len(mode['k_data'])):
                prev_resolved = mode['k_data'][i-1]['is_resolved']
                curr_resolved = mode['k_data'][i]['is_resolved']
                
                if prev_resolved and not curr_resolved:
                    transitions['resolved_to_unresolved'] += 1
                elif not prev_resolved and curr_resolved:
                    transitions['unresolved_to_resolved'] += 1
                elif prev_resolved and curr_resolved:
                    transitions['stable_resolved'] += 1
                else:
                    transitions['stable_unresolved'] += 1
    
    tracked_modes['transitions'] = transitions
    
    print(f"Mode tracking complete. Found {len(tracked_modes['modes'])} distinct modes.")
    print(f"Transitions: {transitions['resolved_to_unresolved']} resolved→unresolved, " +
          f"{transitions['unresolved_to_resolved']} unresolved→resolved")
    
    return tracked_modes

def plot_tracked_modes(tracked_modes, plot_type='real', show_stats=True):
    """
    Plot tracked eigenmode paths as a function of k.
    
    Parameters:
    -----------
    tracked_modes : dict
        Tracked modes information from track_modes()
    plot_type : str
        Type of plot: 'real', 'imag', or 'complex'
    show_stats : bool
        Show mode continuity statistics
    """
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    
    if tracked_modes is None:
        print("No tracked mode data available")
        return
    
    # Extract parameters
    params = tracked_modes['parameters']
    k_values = tracked_modes['k_values']
    modes = tracked_modes['modes']
    
    # Prepare figure based on plot type
    if plot_type == 'complex':
        fig, ax = plt.subplots(1, 2, figsize=(16, 7))
        ax_real = ax[0]
        ax_imag = ax[1]
    else:
        fig, ax = plt.subplots(1, 1, figsize=(10, 7))
        if plot_type == 'real':
            ax_real = ax
            ax_imag = None
        else:
            ax_real = None
            ax_imag = ax
    
    # Create a colormap for mode paths
    colors = cm.rainbow(np.linspace(0, 1, len(modes)))
    
    # For each mode, plot the path
    for i, mode in enumerate(modes):
        k_data = mode['k_data']
        k_vals = [entry['k'] for entry in k_data]
        
        if ax_real is not None:
            real_vals = [entry['eigenvalue'].real for entry in k_data]
            ax_real.plot(k_vals, real_vals, '-o', color=colors[i], 
                        linewidth=1.5, markersize=4, alpha=0.8,
                        label=f"Mode {mode['mode_id']}")
        
        if ax_imag is not None:
            imag_vals = [entry['eigenvalue'].imag for entry in k_data]
            ax_imag.plot(k_vals, imag_vals, '-o', color=colors[i], 
                        linewidth=1.5, markersize=4, alpha=0.8,
                        label=f"Mode {mode['mode_id']}")
    
    # Set titles and labels
    title_base = f"Tracked Modes for m={params['m']}, BV={params['bv_freq']}, ω={params['omega']}"
    
    if plot_type == 'complex':
        ax_real.set_title(f"{title_base} - Real Part")
        ax_real.set_xlabel('k')
        ax_real.set_ylabel('Re(σ)')
        ax_real.grid(True, alpha=0.3)
        
        ax_imag.set_title(f"{title_base} - Imaginary Part")
        ax_imag.set_xlabel('k')
        ax_imag.set_ylabel('Im(σ)')
        ax_imag.grid(True, alpha=0.3)
    elif plot_type == 'real':
        ax_real.set_title(f"{title_base} - Real Part")
        ax_real.set_xlabel('k')
        ax_real.set_ylabel('Re(σ)')
        ax_real.grid(True, alpha=0.3)
    else:
        ax_imag.set_title(f"{title_base} - Imaginary Part")
        ax_imag.set_xlabel('k')
        ax_imag.set_ylabel('Im(σ)')
        ax_imag.grid(True, alpha=0.3)
    
    # If requested, show mode continuity statistics
    if show_stats and 'continuity_stats' in tracked_modes:
        stats = tracked_modes['continuity_stats']
        stats_df = pd.DataFrame({
            'k': list(stats.keys()),
            'Total Modes': [stats[k]['total_modes'] for k in stats],
            'Continuing': [stats[k]['continuing_modes'] for k in stats],
            'New': [stats[k]['new_modes'] for k in stats],
            'Disappearing': [stats[k]['disappearing_modes'] for k in stats]
        })
        stats_df = stats_df.sort_values('k')
        
        from IPython.display import display
        print("Mode Continuity Statistics:")
        display(stats_df)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Example usage:
# tracked_modes = track_modes(all_data, m=2, bv_freq=1.11, omega=-1.2)
# plot_tracked_modes(tracked_modes, plot_type='complex')

In [ ]:
tracked_modes = track_modes(all_data, m=1, bv_freq=1.11, omega=0)
plot_tracked_modes(tracked_modes, plot_type='complex')

In [ ]:
tracked_modes = track_modes(all_data, m=2, bv_freq=1.11, omega=-1.2)
plot_tracked_modes(tracked_modes, plot_type='complex')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import display, clear_output
import time
import numpy as np
import multiprocessing
from multiprocessing import Pool, cpu_count
import os

def compute_similarity_worker(args):
    """Worker function for multiprocessing similarity computation"""
    k1, idx1, k2, idx2, data_dir, bv_freq, omega, m, filtered_data, eig_weight, vel_weight = args
    
    # Get velocity profiles
    vel1 = read_velocity_profile(data_dir, bv_freq, omega, m, k1, idx1)
    vel2 = read_velocity_profile(data_dir, bv_freq, omega, m, k2, idx2)
    
    if vel1 is None or vel2 is None:
        return 0.0
    
    # Find eigenvalues for the modes
    k1_data = next(d for d in filtered_data if abs(d['k'] - k1) < 1e-6)
    k2_data = next(d for d in filtered_data if abs(d['k'] - k2) < 1e-6)
    
    # Get eigenvalue indices
    idx1_pos = np.where(k1_data['indices'] == idx1)[0][0]
    idx2_pos = np.where(k2_data['indices'] == idx2)[0][0]
    
    eig1 = k1_data['eigenvalues'][idx1_pos]
    eig2 = k2_data['eigenvalues'][idx2_pos]
    
    # Compute eigenvalue similarity - inversely proportional to distance
    eig_dist = abs(eig1 - eig2)
    eig_sim = 1.0 / (1.0 + eig_dist)
    
    # Compute velocity profile similarity
    # Interpolate to common r-grid if needed
    r1, r2 = vel1['r'], vel2['r']
    
    # Use common r range
    r_min = max(r1.min(), r2.min())
    r_max = min(r1.max(), r2.max())
    
    # Filter to common range
    mask1 = (r1 >= r_min) & (r1 <= r_max)
    mask2 = (r2 >= r_min) & (r2 <= r_max)
    
    r1_filtered = r1[mask1]
    ur1_real = vel1['ur_real'][mask1]
    uphi1_real = vel1['uphi_real'][mask1]
    uz1_real = vel1['uz_real'][mask1]
    b1_real = vel1['b_real'][mask1]
    
    # Interpolate vel2 to r1_filtered grid
    from scipy.interpolate import interp1d
    
    ur2_real_interp = interp1d(r2[mask2], vel2['ur_real'][mask2], bounds_error=False, fill_value=0)
    uphi2_real_interp = interp1d(r2[mask2], vel2['uphi_real'][mask2], bounds_error=False, fill_value=0)
    uz2_real_interp = interp1d(r2[mask2], vel2['uz_real'][mask2], bounds_error=False, fill_value=0)
    b2_real_interp = interp1d(r2[mask2], vel2['b_real'][mask2], bounds_error=False, fill_value=0)
    
    ur2_real = ur2_real_interp(r1_filtered)
    uphi2_real = uphi2_real_interp(r1_filtered)
    uz2_real = uz2_real_interp(r1_filtered)
    b2_real = b2_real_interp(r1_filtered)
    
    # Compute correlation coefficients for each component
    ur_corr = np.corrcoef(ur1_real, ur2_real)[0, 1]
    uphi_corr = np.corrcoef(uphi1_real, uphi2_real)[0, 1]
    uz_corr = np.corrcoef(uz1_real, uz2_real)[0, 1]
    b_corr = np.corrcoef(b1_real, b2_real)[0, 1]
    
    # Handle NaN values (can happen if arrays are constant)
    ur_corr = 0 if np.isnan(ur_corr) else ur_corr
    uphi_corr = 0 if np.isnan(uphi_corr) else uphi_corr
    uz_corr = 0 if np.isnan(uz_corr) else uz_corr
    b_corr = 0 if np.isnan(b_corr) else b_corr
    
    # Average correlation across all components
    vel_sim = (abs(ur_corr) + abs(uphi_corr) + abs(uz_corr) + abs(b_corr)) / 4.0
    
    # Compute weighted similarity
    total_sim = eig_weight * eig_sim + vel_weight * vel_sim
    
    return total_sim

def track_modes_with_live_plot(data_list, m, bv_freq, omega, k_values=None, sim_threshold=0.8, eig_weight=0.4, vel_weight=0.6, 
                start_with_resolved_only=True, allow_unresolved_continuation=True, plot_type='real', update_delay=0.5,
                use_multiprocessing=True, n_processes=None, min_real_part=0.0, 
                save_interval=10, output_dir=None, max_gap_jumps=2):
    """
    Track eigenvalue modes across different k values with live plotting.
    
    Parameters:
    -----------
    data_list : list
        List of parsed eigenvalue data dictionaries
    m : int
        Azimuthal wavenumber to track
    bv_freq : float
        Brunt-Väisälä frequency
    omega : float
        Background rotation rate
    k_values : list, optional
        Specific k values to track (defaults to all k values for the given m, bv_freq, omega)
    sim_threshold : float
        Similarity threshold for matching modes (0-1)
    eig_weight : float
        Weight for eigenvalue similarity in the matching score (0-1)
    vel_weight : float
        Weight for velocity profile similarity in the matching score (0-1)
    start_with_resolved_only : bool
        If True, only track modes that start as resolved at the first k value
    allow_unresolved_continuation : bool
        If True, allow resolved modes to continue into unresolved modes
    plot_type : str
        Type of plot: 'real', 'imag', or 'complex'
    update_delay : float
        Delay in seconds between plot updates
    use_multiprocessing : bool
        If True, use multiprocessing for similarity computation
    n_processes : int, optional
        Number of processes to use (defaults to CPU count)
    min_real_part : float
        Minimum real part for modes to be considered
    save_interval : int
        Save plots every save_interval k values
    output_dir : str or None
        Directory to save plots (if None, don't save)
    max_gap_jumps : int
        Maximum number of k values a mode can skip when trying to find a continuation
        
    Returns:
    --------
    dict
        Dictionary with tracked modes information
    """
    # Create output directory if needed
    if output_dir is not None:
        import os
        os.makedirs(output_dir, exist_ok=True)
        print(f"Will save plots to {output_dir} every {save_interval} k values")
    
    # Filter data by m, bv_freq, omega
    filtered_data = [d for d in data_list if d['m'] == m and 
                     abs(d['bv_freq'] - bv_freq) < 1e-6 and 
                     abs(d['omega'] - omega) < 1e-6]
    
    if not filtered_data:
        print(f"No data found for m={m}, bv_freq={bv_freq}, omega={omega}")
        return None
    
    # Sort by k value
    filtered_data.sort(key=lambda d: d['k'])
    
    # If k_values is not specified, use all available k values
    if k_values is None:
        k_values = [d['k'] for d in filtered_data]
    else:
        # Filter data further to only include the specified k values
        filtered_data = [d for d in filtered_data if d['k'] in k_values]
        k_values = sorted([d['k'] for d in filtered_data])
    
    if len(k_values) < 2:
        print("At least two k values are needed for mode tracking")
        return None
    
    print(f"Tracking modes for m={m}, bv_freq={bv_freq}, omega={omega} across {len(k_values)} k values")
    print(f"Only considering modes with real part ≥ {min_real_part}")
    print(f"Modes can jump gaps of up to {max_gap_jumps} k values")
    
    if use_multiprocessing:
        n_proc = n_processes or cpu_count()
        print(f"Using multiprocessing with {n_proc} processes")
    
    # Initialize tracked modes dictionary
    tracked_modes = {
        'parameters': {
            'm': m,
            'bv_freq': bv_freq,
            'omega': omega,
            'min_real_part': min_real_part,
            'max_gap_jumps': max_gap_jumps,
        },
        'k_values': k_values,
        'modes': []
    }
    
    # Get data for the first k value
    first_k_data = next(d for d in filtered_data if d['k'] == k_values[0])
    first_k_eigvals = first_k_data['eigenvalues']
    first_k_indices = first_k_data['indices']
    first_k_resolved = first_k_data['is_resolved']
    
    # Initialize modes with data from the first k value
    # If start_with_resolved_only is True, only include resolved modes
    # Also filter for modes with real part >= min_real_part
    for i, (eig, idx, resolved) in enumerate(zip(first_k_eigvals, first_k_indices, first_k_resolved)):
        if (not start_with_resolved_only or resolved) and eig.real >= min_real_part:
            # Initialize a new mode with the first k value
            new_mode = {
                'k_data': [{
                    'k': k_values[0],
                    'eigenvalue': eig,
                    'index': int(idx),
                    'data_index': i,
                    'is_resolved': resolved
                }],
                'mode_id': len(tracked_modes['modes']),
                'k_coverage': [k_values[0]],
                'last_matched_k_idx': 0,  # Index in k_values where this mode was last matched
            }
            tracked_modes['modes'].append(new_mode)
    
    print(f"Starting with {len(tracked_modes['modes'])} " + 
          ("resolved " if start_with_resolved_only else "") + 
          f"modes at k={k_values[0]} with real part ≥ {min_real_part}")
    
    # Create a function to generate and display/save the current state plot
    def generate_plot():
        # Create figure and axes based on plot type
        if plot_type == 'complex':
            fig, ax = plt.subplots(1, 2, figsize=(16, 7))
            ax_real, ax_imag = ax
        else:
            fig, ax = plt.subplots(1, 1, figsize=(10, 7))
            if plot_type == 'real':
                ax_real, ax_imag = ax, None
            else:
                ax_real, ax_imag = None, ax
        
        # Create a colormap for mode paths
        colors = cm.rainbow(np.linspace(0, 1, max(len(tracked_modes['modes']), 1)))
        
        # Plot each mode
        for i, mode in enumerate(tracked_modes['modes']):
            k_data = mode['k_data']
            k_vals = [entry['k'] for entry in k_data]
            
            if ax_real is not None:
                real_vals = [entry['eigenvalue'].real for entry in k_data]
                ax_real.plot(k_vals, real_vals, '-o', color=colors[i % len(colors)], 
                            linewidth=1.5, markersize=4, alpha=0.8,
                            label=f"Mode {mode['mode_id']}")
            
            if ax_imag is not None:
                imag_vals = [entry['eigenvalue'].imag for entry in k_data]
                ax_imag.plot(k_vals, imag_vals, '-o', color=colors[i % len(colors)], 
                            linewidth=1.5, markersize=4, alpha=0.8,
                            label=f"Mode {mode['mode_id']}")
        
        # Set titles and labels
        title_base = f"Tracked Modes for m={m}, BV={bv_freq}, ω={omega}"
        
        if plot_type == 'complex':
            ax_real.set_title(f"{title_base} - Real Part")
            ax_real.set_xlabel('k')
            ax_real.set_ylabel('Re(σ)')
            ax_real.grid(True, alpha=0.3)
            
            ax_imag.set_title(f"{title_base} - Imaginary Part")
            ax_imag.set_xlabel('k')
            ax_imag.set_ylabel('Im(σ)')
            ax_imag.grid(True, alpha=0.3)
        elif plot_type == 'real':
            ax_real.set_title(f"{title_base} - Real Part")
            ax_real.set_xlabel('k')
            ax_real.set_ylabel('Re(σ)')
            ax_real.grid(True, alpha=0.3)
        else:
            ax_imag.set_title(f"{title_base} - Imaginary Part")
            ax_imag.set_xlabel('k')
            ax_imag.set_ylabel('Im(σ)')
            ax_imag.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    # Try both interactive mode and saving at intervals
    try:
        # Set interactive mode
        plt.ion()
        
        # Initial plot
        fig = generate_plot()
        plt.show()
        
        # Function to compute similarity between eigenmodes (non-multiprocessing version)
        def compute_similarity(k1, idx1, k2, idx2):
            """Compute similarity between two eigenmodes"""
            return compute_similarity_worker((k1, idx1, k2, idx2, data_dir, bv_freq, omega, m, 
                                            filtered_data, eig_weight, vel_weight))
        
        # Track modes across k values
        for k_idx in range(1, len(k_values)):
            curr_k = k_values[k_idx]
            prev_k = k_values[k_idx - 1]
            
            print(f"Tracking from k={prev_k} to k={curr_k}... ({k_idx}/{len(k_values)-1})")
            
            # Get data for the current k value
            curr_k_data = next(d for d in filtered_data if d['k'] == curr_k)
            curr_k_eigvals = curr_k_data['eigenvalues']
            curr_k_indices = curr_k_data['indices']
            curr_k_resolved = curr_k_data['is_resolved']
            
            # Create a list of available modes for the current k value
            # Include all modes regardless of resolved status if we're allowing unresolved continuation
            # Also filter by minimum real part
            available_modes = [(i, idx, eig, resolved) for i, (eig, idx, resolved) 
                             in enumerate(zip(curr_k_eigvals, curr_k_indices, curr_k_resolved))
                             if eig.real >= min_real_part]  # Filter by minimum real part
            
            # List to keep track of which modes have been matched
            matched_curr_modes = [False] * len(available_modes)
            
            # For each previously tracked mode, find the best match
            for mode in tracked_modes['modes']:
                # Check if this mode can potentially continue at the current k_idx
                # The mode must have been matched previously, and not already matched at this k_idx
                last_matched_idx = mode.get('last_matched_k_idx', 0)
                
                # Skip if the gap is too large
                if k_idx - last_matched_idx > max_gap_jumps + 1:
                    continue
                
                # Skip if this mode already has a match at this k
                if curr_k in mode['k_coverage']:
                    continue
                    
                # Get the last entry for this mode
                last_entry = mode['k_data'][-1]
                last_k = last_entry['k']
                last_idx = last_entry['index']
                last_resolved = last_entry['is_resolved']
                
                # Prepare similarity computation tasks
                valid_curr_modes = []
                for i, (curr_i, curr_idx, curr_eig, curr_resolved) in enumerate(available_modes):
                    if not matched_curr_modes[i]:  # Only consider unmatched modes
                        # If we don't allow unresolved continuation and this mode is unresolved, skip it
                        if not allow_unresolved_continuation and not curr_resolved:
                            continue
                        valid_curr_modes.append((i, curr_i, curr_idx, curr_eig, curr_resolved))
                
                if valid_curr_modes:
                    if use_multiprocessing and len(valid_curr_modes) > 1:
                        try:
                            # Use multiprocessing for similarity computation
                            similarity_args = []
                            for i, curr_i, curr_idx, curr_eig, curr_resolved in valid_curr_modes:
                                args = (last_k, last_idx, curr_k, curr_idx, data_dir, bv_freq, omega, m,
                                       filtered_data, eig_weight, vel_weight)
                                similarity_args.append(args)
                            
                            # Set start method to 'fork' on Unix systems for better performance
                            # This is needed on macOS to avoid issues with pickle
                            if hasattr(multiprocessing, 'get_context'):
                                ctx = multiprocessing.get_context('fork' if os.name != 'nt' else 'spawn')
                                with ctx.Pool(processes=n_processes) as pool:
                                    similarities = pool.map(compute_similarity_worker, similarity_args)
                            else:
                                # Fallback for older Python versions
                                with Pool(processes=n_processes) as pool:
                                    similarities = pool.map(compute_similarity_worker, similarity_args)
                            
                            # Combine results with indices
                            similarities = [(valid_curr_modes[j][0], sim) for j, sim in enumerate(similarities)]
                        except Exception as e:
                            print(f"Multiprocessing failed: {e}. Falling back to single-threaded computation.")
                            use_multiprocessing = False
                            similarities = []
                            for i, curr_i, curr_idx, curr_eig, curr_resolved in valid_curr_modes:
                                sim = compute_similarity(last_k, last_idx, curr_k, curr_idx)
                                similarities.append((i, sim))
                    else:
                        # Use single-threaded computation
                        similarities = []
                        for i, curr_i, curr_idx, curr_eig, curr_resolved in valid_curr_modes:
                            sim = compute_similarity(last_k, last_idx, curr_k, curr_idx)
                            similarities.append((i, sim))
                    
                    # Find the best match above the threshold
                    if similarities:
                        best_match = max(similarities, key=lambda x: x[1])
                        best_idx, best_sim = best_match
                        
                        if best_sim >= sim_threshold:
                            # Match found, update the mode
                            curr_i, curr_idx, curr_eig, curr_resolved = available_modes[best_idx]
                            
                            # If this is a gap jump, add interpolated points
                            if k_idx - last_matched_idx > 1:
                                # Calculate number of skipped k values
                                n_skips = k_idx - last_matched_idx - 1
                                skipped_k_indices = range(last_matched_idx + 1, k_idx)
                                
                                # Interpolate eigenvalues for skipped k values
                                last_eig = last_entry['eigenvalue']
                                for j, skip_k_idx in enumerate(skipped_k_indices, 1):
                                    skip_k = k_values[skip_k_idx]
                                    # Linear interpolation of eigenvalue
                                    t = j / (n_skips + 1)  # interpolation parameter [0, 1]
                                    interp_eig = last_eig * (1 - t) + curr_eig * t
                                    
                                    # Add interpolated point (marked as not resolved)
                                    mode['k_data'].append({
                                        'k': skip_k,
                                        'eigenvalue': interp_eig,
                                        'index': -1,  # Special index to mark interpolated data
                                        'data_index': -1,
                                        'is_resolved': False,
                                        'interpolated': True
                                    })
                                    mode['k_coverage'].append(skip_k)
                                
                                # Add a message about the gap jump
                                gap_msg = f"Mode {mode['mode_id']} jumped {n_skips} k-value(s) " + \
                                         f"from k={last_k} to k={curr_k} with similarity {best_sim:.3f}"
                                print(gap_msg)
                            
                            # Now add the current point
                            mode['k_data'].append({
                                'k': curr_k,
                                'eigenvalue': curr_eig,
                                'index': int(curr_idx),
                                'data_index': curr_i,
                                'is_resolved': curr_resolved
                            })
                            mode['k_coverage'].append(curr_k)
                            mode['last_matched_k_idx'] = k_idx
                            matched_curr_modes[best_idx] = True
                            
                            # Print status including resolved/unresolved transition info
                            status = f"Mode {mode['mode_id']} continued with similarity {best_sim:.3f}"
                            if last_resolved and not curr_resolved:
                                status += " (resolved → unresolved)"
                            elif not last_resolved and curr_resolved:
                                status += " (unresolved → resolved)"
                            print(status)
            
            # Check for any new resolved modes that appeared at this k value
            # Only add new modes if they're resolved (we're already tracking the important ones)
            for i, matched in enumerate(matched_curr_modes):
                if not matched:
                    curr_i, curr_idx, curr_eig, curr_resolved = available_modes[i]
                    
                    # Only add new modes if they're resolved
                    if curr_resolved:
                        # Create a new mode
                        new_mode = {
                            'k_data': [{
                                'k': curr_k,
                                'eigenvalue': curr_eig,
                                'index': int(curr_idx),
                                'data_index': curr_i,
                                'is_resolved': curr_resolved
                            }],
                            'mode_id': len(tracked_modes['modes']),
                            'k_coverage': [curr_k],
                            'last_matched_k_idx': k_idx
                        }
                        tracked_modes['modes'].append(new_mode)
                        print(f"New resolved mode appeared with ID {new_mode['mode_id']} at k={curr_k}")
            
            # Update plot
            if k_idx % save_interval == 0 or k_idx == len(k_values) - 1:
                # Close previous figure to avoid memory issues
                plt.close(fig)
                
                # Generate new plot
                fig = generate_plot()
                
                # Save plot if directory is provided
                if output_dir is not None:
                    filename = f"mode_tracking_m{m}_bv{bv_freq}_w{omega}_k{curr_k:.6f}.png"
                    filepath = os.path.join(output_dir, filename)
                    fig.savefig(filepath, dpi=300, bbox_inches='tight')
                    print(f"Saved plot to {filepath}")
                
                # Try to display the plot (for interactive mode)
                plt.draw()
                plt.pause(0.1)  # Slightly longer pause to ensure display
                
            # Additional delay if specified
            if update_delay > 0:
                time.sleep(update_delay)
        
        # Turn off interactive mode
        plt.ioff()
        
    except Exception as e:
        print(f"Error during interactive plotting: {e}")
        print("Falling back to static plot generation...")
    
    # Compute continuity statistics
    continuity_stats = {}
    for k_value in k_values:
        continuity_stats[k_value] = {
            'total_modes': len([m for m in tracked_modes['modes'] if k_value in m['k_coverage']]),
            'resolved_modes': len([m for m in tracked_modes['modes'] 
                                 if k_value in m['k_coverage'] and 
                                 next(entry for entry in m['k_data'] if entry['k'] == k_value).get('is_resolved', False)]),
            'unresolved_modes': len([m for m in tracked_modes['modes'] 
                                   if k_value in m['k_coverage'] and 
                                   not next(entry for entry in m['k_data'] if entry['k'] == k_value).get('is_resolved', False)]),
            'interpolated_points': len([m for m in tracked_modes['modes'] 
                                    if k_value in m['k_coverage'] and 
                                    next(entry for entry in m['k_data'] if entry['k'] == k_value).get('interpolated', False)]),
            'continuing_modes': 0,
            'new_modes': 0,
            'disappearing_modes': 0
        }
    
    for mode in tracked_modes['modes']:
        k_coverage = mode['k_coverage']
        for i, k in enumerate(k_values):
            if k in k_coverage:
                if i > 0 and k_values[i-1] in k_coverage:
                    continuity_stats[k]['continuing_modes'] += 1
                else:
                    continuity_stats[k]['new_modes'] += 1
                if i < len(k_values)-1 and k_values[i+1] not in k_coverage:
                    continuity_stats[k]['disappearing_modes'] += 1
    
    tracked_modes['continuity_stats'] = continuity_stats
    
    # Calculate statistics for gap jumps
    gap_jumps = {
        'modes_with_jumps': 0,
        'total_jumps': 0,
        'jump_lengths': {}  # Count of jumps by length
    }
    
    for mode in tracked_modes['modes']:
        mode_jumps = 0
        prev_k_idx = None
        
        # Find the k_idx for each k value in the mode's coverage
        k_indices = [k_values.index(k) for k in mode['k_coverage']]
        k_indices.sort()
        
        for k_idx in k_indices:
            if prev_k_idx is not None and k_idx - prev_k_idx > 1:
                # Found a gap
                jump_length = k_idx - prev_k_idx - 1
                mode_jumps += 1
                gap_jumps['total_jumps'] += 1
                
                # Update jump length histogram
                if jump_length not in gap_jumps['jump_lengths']:
                    gap_jumps['jump_lengths'][jump_length] = 0
                gap_jumps['jump_lengths'][jump_length] += 1
                
            prev_k_idx = k_idx
            
        if mode_jumps > 0:
            gap_jumps['modes_with_jumps'] += 1
    
    tracked_modes['gap_jumps'] = gap_jumps
    
    # Calculate resolved/unresolved transition statistics
    transitions = {
        'resolved_to_unresolved': 0,
        'unresolved_to_resolved': 0,
        'stable_resolved': 0,
        'stable_unresolved': 0
    }
    
    for mode in tracked_modes['modes']:
        if len(mode['k_data']) >= 2:
            for i in range(1, len(mode['k_data'])):
                # Skip transitions involving interpolated points
                if mode['k_data'][i-1].get('interpolated', False) or mode['k_data'][i].get('interpolated', False):
                    continue
                    
                prev_resolved = mode['k_data'][i-1]['is_resolved']
                curr_resolved = mode['k_data'][i]['is_resolved']
                
                if prev_resolved and not curr_resolved:
                    transitions['resolved_to_unresolved'] += 1
                elif not prev_resolved and curr_resolved:
                    transitions['unresolved_to_resolved'] += 1
                elif prev_resolved and curr_resolved:
                    transitions['stable_resolved'] += 1
                else:
                    transitions['stable_unresolved'] += 1
    
    tracked_modes['transitions'] = transitions
    
    # Print overall statistics
    print(f"\nMode tracking complete. Found {len(tracked_modes['modes'])} distinct modes.")
    print(f"Transitions: {transitions['resolved_to_unresolved']} resolved→unresolved, " +
          f"{transitions['unresolved_to_resolved']} unresolved→resolved")
    
    # Print gap jump statistics
    print(f"Modes with gap jumps: {gap_jumps['modes_with_jumps']} of {len(tracked_modes['modes'])}")
    print(f"Total gap jumps: {gap_jumps['total_jumps']}")
    
    if gap_jumps['jump_lengths']:
        print("Jump length histogram:")
        for length, count in sorted(gap_jumps['jump_lengths'].items()):
            print(f"  {length} k-value(s): {count} jump(s)")
    
    # Create a final static plot
    final_fig = generate_plot()
    plt.show()
    
    # Save final plot if directory is specified
    if output_dir is not None:
        filename = f"mode_tracking_m{m}_bv{bv_freq}_w{omega}_final.png"
        filepath = os.path.join(output_dir, filename)
        final_fig.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"Saved final plot to {filepath}")
    
    return tracked_modes

In [ ]:
# tracked_result = track_modes_with_live_plot(all_data, m=1, bv_freq=1.11, omega=0, k_values=None, sim_threshold=0.8, eig_weight=0.4, vel_weight=0.6,
#                 start_with_resolved_only=True, allow_unresolved_continuation=True, plot_type='complex', update_delay=0.5,
#                 use_multiprocessing=True, n_processes=None)

output_dir = os.path.join(os.getcwd(), "mode_tracking_plots")
tracked_result = track_modes_with_live_plot(
    all_data, 
    m=1, 
    bv_freq=1.11, 
    omega=0, 
    k_values=None, 
    sim_threshold=0.8, 
    eig_weight=0.4, 
    vel_weight=0.6,
    start_with_resolved_only=True, 
    allow_unresolved_continuation=True, 
    plot_type='complex', 
    update_delay=0.5,
    use_multiprocessing=True, 
    n_processes=None,
    min_real_part=-1e-6,  # Only consider modes with non-negative real parts
    save_interval=10,  # Save a plot every 10 k values
    output_dir=output_dir,  # Directory to save plots
    max_gap_jumps=2  # Allow modes to jump gaps of up to 2 k values
)

In [ ]:
plot_tracked_modes(tracked_result, plot_type='complex')

## Mode tracking 2

In [ ]:
def create_mode_tracking_interactive(data_list, m_value=None, omega=None, bv_freq=None, 
                                     eig_min=0, eig_max=0.01, sim_threshold=0.7,
                                     eig_weight=0.4, vel_weight=0.6, max_gap_jumps=2):
    """
    Create an interactive dispersion plot that allows tracking modes when clicking on points.
    
    Parameters:
    -----------
    data_list : list
        List of parsed eigenvalue data dictionaries
    m_value : int or None
        Azimuthal wavenumber filter (if None, will use the first available m)
    omega : float or None
        Rotation rate filter (if None, will use all omega values)
    bv_freq : float or None
        Brunt-Väisälä frequency filter (if None, will use all bv_freq values)
    eig_min, eig_max : float
        Y-axis range for the real part plot
    sim_threshold : float
        Similarity threshold for matching modes (0-1)
    eig_weight : float
        Weight for eigenvalue similarity in the matching score
    vel_weight : float
        Weight for velocity profile similarity in the matching score
    max_gap_jumps : int
        Maximum number of k values a mode can skip when trying to find a continuation
    
    Returns:
    --------
    VBox
        Widget containing the interactive plot
    """
    # Filter data by bv_freq if specified
    if bv_freq is not None:
        data_list = [d for d in data_list if d['bv_freq'] == bv_freq]
    # Filter data by omega if specified
    if omega is not None:
        data_list = [d for d in data_list if d['omega'] == omega]
    
    # Get unique m values for the dropdown
    m_values = sorted(list(set(d['m'] for d in data_list)))
    
    if m_value is None and m_values:
        m_value = m_values[0]
    
    # Prepare output widgets
    out_disp = Output()
    out_tracking = Output()
    
    # Create dropdown for m values
    m_dropdown = Dropdown(
        options=[(f'm={m}', m) for m in m_values],
        value=m_value,
        description='Mode:',
        layout=Layout(width='200px')
    )
    
    # Pre-process data by m value
    data_by_m = {}
    for m in m_values:
        group = [d for d in data_list if d['m'] == m]
        group.sort(key=lambda x: x['k'])
        data_by_m[m] = group
        
    # Pre-process points data
    points_by_m = {}
    for m, group in data_by_m.items():
        if not group:
            continue
            
        Fr = 1 / group[0]['bv_freq']
        points_data = []
        for d in group:
            eigenvalues = d['eigenvalues']
            is_resolved = d['is_resolved']
            k_Fr = d['k'] * Fr
            indices = d['indices']
            for i, (eig, idx, resolved) in enumerate(zip(eigenvalues, indices, is_resolved)):
                point_data = {
                    'real': eig.real,
                    'imag': eig.imag,
                    'k_Fr': k_Fr,
                    'bv_freq': d['bv_freq'],
                    'omega': d['omega'],
                    'm': d['m'],
                    'k': d['k'],
                    'index': int(idx),
                    'resolved': resolved,
                    'data_idx': i
                }
                points_data.append(point_data)
        points_by_m[m] = points_data

    # Function to create plot for a given m value
    def create_plot_for_m(m):
        if m not in points_by_m or not points_by_m[m]:
            with out_disp:
                out_disp.clear_output(wait=True)
                print(f"No data found for m={m}")
            return None, None
        
        points_data = points_by_m[m]
        
        # Extract data for plotting
        k_Fr_resolved = [pt['k_Fr'] for pt in points_data if pt['resolved']]
        real_resolved = [pt['real'] for pt in points_data if pt['resolved']]
        imag_resolved = [pt['imag'] for pt in points_data if pt['resolved']]
        
        k_Fr_unresolved = [pt['k_Fr'] for pt in points_data if not pt['resolved']]
        real_unresolved = [pt['real'] for pt in points_data if not pt['resolved']]
        imag_unresolved = [pt['imag'] for pt in points_data if not pt['resolved']]
        
        # Create subplots
        fig = make_subplots(rows=1, cols=2, 
                          subplot_titles=("Real Part", "Imaginary Part"),
                          horizontal_spacing=0.1)
        
        # Add traces for real part
        fig.add_trace(
            go.Scatter(
                x=k_Fr_unresolved, y=real_unresolved,
                mode='markers', name='Unresolved',
                marker=dict(size=3, color='blue', opacity=1),
                customdata=[{
                    'k': points_data[i]['k'],
                    'index': points_data[i]['index'],
                    'real': points_data[i]['real'],
                    'imag': points_data[i]['imag'],
                    'm': points_data[i]['m'],
                    'bv_freq': points_data[i]['bv_freq'],
                    'omega': points_data[i]['omega'],
                    'data_idx': points_data[i]['data_idx'],
                    'point_idx': i
                } for i, pt in enumerate(points_data) if not pt['resolved']],
                hovertemplate='k*Fr: %{x:.3f}<br>Re(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=k_Fr_resolved, y=real_resolved,
                mode='markers', name='Resolved',
                marker=dict(size=5, color='red', opacity=1),
                customdata=[{
                    'k': points_data[i]['k'],
                    'index': points_data[i]['index'],
                    'real': points_data[i]['real'],
                    'imag': points_data[i]['imag'],
                    'm': points_data[i]['m'],
                    'bv_freq': points_data[i]['bv_freq'],
                    'omega': points_data[i]['omega'],
                    'data_idx': points_data[i]['data_idx'],
                    'point_idx': i
                } for i, pt in enumerate(points_data) if pt['resolved']],
                hovertemplate='k*Fr: %{x:.3f}<br>Re(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=1
        )
        
        # Add traces for imaginary part
        fig.add_trace(
            go.Scatter(
                x=k_Fr_unresolved, y=imag_unresolved,
                mode='markers', name='Unresolved',
                marker=dict(size=3, color='blue', opacity=1),
                customdata=[{
                    'k': points_data[i]['k'],
                    'index': points_data[i]['index'],
                    'real': points_data[i]['real'],
                    'imag': points_data[i]['imag'],
                    'm': points_data[i]['m'],
                    'bv_freq': points_data[i]['bv_freq'],
                    'omega': points_data[i]['omega'],
                    'data_idx': points_data[i]['data_idx'],
                    'point_idx': i
                } for i, pt in enumerate(points_data) if not pt['resolved']],
                hovertemplate='k*Fr: %{x:.3f}<br>Im(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=2
        )

        fig.add_trace(
            go.Scatter(
                x=k_Fr_resolved, y=imag_resolved,
                mode='markers', name='Resolved',
                marker=dict(size=5, color='red', opacity=1),
                customdata=[{
                    'k': points_data[i]['k'],
                    'index': points_data[i]['index'],
                    'real': points_data[i]['real'],
                    'imag': points_data[i]['imag'],
                    'm': points_data[i]['m'],
                    'bv_freq': points_data[i]['bv_freq'],
                    'omega': points_data[i]['omega'],
                    'data_idx': points_data[i]['data_idx'],
                    'point_idx': i
                } for i, pt in enumerate(points_data) if pt['resolved']],
                hovertemplate='k*Fr: %{x:.3f}<br>Im(σ): %{y:.6f}<br>k: %{customdata.k}<br>index: %{customdata.index}<extra></extra>'
            ),
            row=1, col=2
        )
        
        # Pre-create highlight traces (hidden)
        fig.add_scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color='rgba(0,0,0,0)', 
                       line=dict(color='green', width=3)),
            showlegend=False, 
            name='Highlight Real',
            hoverinfo='none',
            row=1, col=1
        )
        
        fig.add_scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(size=8, color='rgba(0,0,0,0)', 
                       line=dict(color='green', width=3)),
            showlegend=False, 
            name='Highlight Imag',
            hoverinfo='none',
            row=1, col=2
        )
        
        # Add traces for tracked mode path (initially empty)
        fig.add_scatter(
            x=[], y=[],
            mode='lines+markers',
            line=dict(color='green', width=2),
            marker=dict(size=6),
            name='Tracked Path (Real)',
            showlegend=True,
            row=1, col=1
        )
        
        fig.add_scatter(
            x=[], y=[],
            mode='lines+markers',
            line=dict(color='green', width=2),
            marker=dict(size=6),
            name='Tracked Path (Imag)',
            showlegend=True,
            row=1, col=2
        )

        # Update layout
        bv = data_by_m[m][0]['bv_freq']
        w = data_by_m[m][0]['omega']
        fig.update_layout(
            title_text=f'Dispersion Relations & Mode Tracking (BV={bv}, Ω̄={w}, m={m})',
            height=800,
            width=1600,
            clickmode='event',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            )
        )
        
        all_k_Fr = k_Fr_resolved + k_Fr_unresolved
        k_min, k_max = min(all_k_Fr) if all_k_Fr else 0, max(all_k_Fr) if all_k_Fr else 1
        
        fig.update_xaxes(title_text='k*Fr', row=1, col=1)
        fig.update_xaxes(title_text='k*Fr', row=1, col=2)
        fig.update_yaxes(title_text='Re(σ)', row=1, col=1)
        fig.update_yaxes(title_text='Im(σ)', row=1, col=2)
        
        # Set y range for real part to show details near zero
        fig.update_yaxes(range=[eig_min, eig_max], row=1, col=1)
        
        # Set x range based on data
        fig.update_xaxes(range=[k_min - 0.5, k_max + 0.5], row=1, col=1)
        fig.update_xaxes(range=[k_min - 0.5, k_max + 0.5], row=1, col=2)
        
        return fig, points_data, data_by_m[m]
    
    def compute_similarity(mode1, mode2, filtered_data=None, eig_weight=0.4, vel_weight=0.6, r_range=None):
        """
        Compute similarity between two eigenmodes with phase normalization and proper complex comparison
        
        Parameters:
        -----------
        mode1, mode2 : dict
            Mode dictionaries containing eigenvalue and metadata
        filtered_data : unused (kept for compatibility)
        eig_weight : float
            Weight for eigenvalue similarity
        vel_weight : float
            Weight for velocity profile similarity
        r_range : tuple or None
            Optional (r_min, r_max) to restrict comparison range
        
        Returns:
        --------
        float
            Similarity score between 0 and 1
        """
        k1 = mode1['k']
        idx1 = mode1['index']
        k2 = mode2['k']
        idx2 = mode2['index']
        m = mode1['m']
        bv_freq = mode1['bv_freq']
        omega = mode1['omega']
        
        # Get velocity profiles
        vel1 = read_velocity_profile(data_dir, bv_freq, omega, m, k1, idx1)
        vel2 = read_velocity_profile(data_dir, bv_freq, omega, m, k2, idx2)
        
        if vel1 is None or vel2 is None:
            # Fall back to eigenvalue-only comparison
            eig1 = complex(mode1['real'], mode1['imag'])
            eig2 = complex(mode2['real'], mode2['imag'])
            eig_dist = abs(eig1 - eig2)
            return 1.0 / (1.0 + eig_dist)
        
        # Compute eigenvalue similarity - inversely proportional to distance
        eig1 = complex(mode1['real'], mode1['imag'])
        eig2 = complex(mode2['real'], mode2['imag'])
        eig_dist = abs(eig1 - eig2)
        eig_sim = 1.0 / (1.0 + eig_dist)
        
        # Get radial coordinates
        r = vel1['r']
        
        # Apply range filter if specified
        if r_range is not None:
            r_min, r_max = r_range
            mask = (r >= r_min) & (r <= r_max)
            r = r[mask]
        else:
            mask = slice(None)  # Use all points
        
        # Get complex velocity fields (already on same grid)
        ur1 = vel1['ur'][mask]
        uphi1 = vel1['uphi'][mask]
        uz1 = vel1['uz'][mask]
        
        ur2 = vel2['ur'][mask]
        uphi2 = vel2['uphi'][mask]
        uz2 = vel2['uz'][mask]
        
        # Phase normalization: Find maximum amplitude position in ur component
        ur1_amp = np.abs(ur1)
        ur2_amp = np.abs(ur2)
        
        if np.max(ur1_amp) > 1e-10:
            # Find index of maximum amplitude for ur1
            max_idx1 = np.argmax(ur1_amp)
            # Get phase at that position
            phase1 = np.angle(ur1[max_idx1])
            # Apply phase rotation to make the maximum point real and positive
            phase_correction1 = -phase1
            
            # Apply same phase correction to all fields
            ur1 = ur1 * np.exp(1j * phase_correction1)
            uphi1 = uphi1 * np.exp(1j * phase_correction1)
            uz1 = uz1 * np.exp(1j * phase_correction1)
        
        if np.max(ur2_amp) > 1e-10:
            # Same normalization for profile 2
            max_idx2 = np.argmax(ur2_amp)
            phase2 = np.angle(ur2[max_idx2])
            phase_correction2 = -phase2
            
            ur2 = ur2 * np.exp(1j * phase_correction2)
            uphi2 = uphi2 * np.exp(1j * phase_correction2)
            uz2 = uz2 * np.exp(1j * phase_correction2)
        
        # Direct complex vector similarity using normalized inner product
        def complex_similarity(a, b):
            # Compute normalized inner product between two complex vectors
            # Returns a value between 0 (orthogonal) and 1 (parallel)
            a_norm = np.linalg.norm(a)
            b_norm = np.linalg.norm(b)
            
            if a_norm < 1e-10 or b_norm < 1e-10:
                return 0.0
                
            # Normalized inner product (absolute value ensures phase insensitivity)
            return abs(np.sum(a.conjugate() * b) / (a_norm * b_norm))
        
        # Compute similarities for each component
        ur_sim = complex_similarity(ur1, ur2)
        uphi_sim = complex_similarity(uphi1, uphi2)
        uz_sim = complex_similarity(uz1, uz2)
        
        # Average velocity similarity across all components
        vel_sim = (ur_sim + uphi_sim + uz_sim ) / 3.0
        
        # Compute weighted similarity
        total_sim = eig_weight * eig_sim + vel_weight * vel_sim
        
        return total_sim

    # Function to track a mode given a starting point
    def track_mode(start_point, k_values, all_data_points, current_fig_widget):
        """
        Track a mode from a starting point across available k values in both directions.
        
        Parameters:
        -----------
        start_point : dict
            Dictionary with data for the starting eigenmode
        k_values : list
            Sorted list of all k values
        all_data_points : list
            List of all data points
        current_fig_widget : go.FigureWidget
            Current figure widget to update
        """
        # Find the start point in k_values
        start_k = start_point['k']
        start_idx = k_values.index(start_k)
        
        # Initialize tracked mode with the starting point
        tracked_mode = {
            'points': [start_point],
            'k_values': [start_k],
            'similarity_scores': []
        }
        
        with out_tracking:
            out_tracking.clear_output(wait=True)
            print(f"Starting mode tracking from k={start_k:.4f}, index={start_point['index']}")
            print(f"Eigenvalue: ({start_point['real']:.6f}, {start_point['imag']:.6f})")
            if start_point['resolved']:
                print("Starting point is resolved")
            else:
                print("Starting point is unresolved")
        
        # Group data by k values for faster lookup
        data_by_k = {}
        for pt in all_data_points:
            k = pt['k']
            if k not in data_by_k:
                data_by_k[k] = []
            data_by_k[k].append(pt)
            
        # Track forward (increasing k)
        current_point = start_point
        skipped_forward = 0
        
        for k_idx in range(start_idx + 1, len(k_values)):
            # Allow skipping up to max_gap_jumps consecutive k values
            if skipped_forward >= max_gap_jumps:
                break
                
            curr_k = k_values[k_idx]
            
            # Check if we have data for this k value
            if curr_k not in data_by_k:
                skipped_forward += 1
                continue
                
            # Find best match
            candidates = data_by_k[curr_k]
            similarities = []
            
            for candidate in candidates:
                sim = compute_similarity(current_point, candidate, None, 
                                        eig_weight=eig_weight, vel_weight=vel_weight, r_range=(0, 20))
                similarities.append((candidate, sim))
            
            if similarities:
                best_match = max(similarities, key=lambda x: x[1])
                best_candidate, best_sim = best_match
                
                if best_sim >= sim_threshold:
                    tracked_mode['points'].append(best_candidate)
                    tracked_mode['k_values'].append(curr_k)
                    tracked_mode['similarity_scores'].append(best_sim)
                    current_point = best_candidate
                    skipped_forward = 0  # Reset skipped count
                    with out_tracking:
                        print(f"Matched forward to k={curr_k:.4f}, index={best_candidate['index']} (similarity: {best_sim:.3f})")
                else:
                    skipped_forward += 1
                    with out_tracking:
                        print(f"No match above threshold for k={curr_k:.4f} (best: {best_sim:.3f})")
            else:
                skipped_forward += 1
        
        # Track backward (decreasing k)
        current_point = start_point
        skipped_backward = 0
        
        for k_idx in range(start_idx - 1, -1, -1):
            # Allow skipping up to max_gap_jumps consecutive k values
            if skipped_backward >= max_gap_jumps:
                break
                
            curr_k = k_values[k_idx]
            
            # Check if we have data for this k value
            if curr_k not in data_by_k:
                skipped_backward += 1
                continue
                
            # Find best match
            candidates = data_by_k[curr_k]
            similarities = []
            
            for candidate in candidates:
                sim = compute_similarity(current_point, candidate, None, 
                                        eig_weight=eig_weight, vel_weight=vel_weight, r_range=(0, 20))
                similarities.append((candidate, sim))
            
            if similarities:
                best_match = max(similarities, key=lambda x: x[1])
                best_candidate, best_sim = best_match
                
                if best_sim >= sim_threshold:
                    tracked_mode['points'].insert(0, best_candidate)
                    tracked_mode['k_values'].insert(0, curr_k)
                    tracked_mode['similarity_scores'].insert(0, best_sim)
                    current_point = best_candidate
                    skipped_backward = 0  # Reset skipped count
                    with out_tracking:
                        print(f"Matched backward to k={curr_k:.4f}, index={best_candidate['index']} (similarity: {best_sim:.3f})")
                else:
                    skipped_backward += 1
                    with out_tracking:
                        print(f"No match above threshold for k={curr_k:.4f} (best: {best_sim:.3f})")
            else:
                skipped_backward += 1
        
        # Print summary
        with out_tracking:
            print(f"\nTracking complete. Found {len(tracked_mode['k_values'])} points across k values.")
            print(f"k range: {min(tracked_mode['k_values']):.4f} to {max(tracked_mode['k_values']):.4f}")
            
            # Count resolved/unresolved
            resolved_count = sum(1 for pt in tracked_mode['points'] if pt['resolved'])
            print(f"Resolved points: {resolved_count}/{len(tracked_mode['points'])}")
            
            if len(tracked_mode['similarity_scores']) > 0:
                print(f"Average similarity score: {np.mean(tracked_mode['similarity_scores']):.3f}")
        
        # Update the plot with tracked path
        Fr = 1 / start_point['bv_freq']  # Get Froude number
        
        # Extract data for plotting
        k_Fr_values = [k * Fr for k in tracked_mode['k_values']]
        real_values = [pt['real'] for pt in tracked_mode['points']]
        imag_values = [pt['imag'] for pt in tracked_mode['points']]
        
        # Update tracked path traces
        with current_fig_widget.batch_update():
            # Real part trace (index 6)
            current_fig_widget.data[6].x = k_Fr_values
            current_fig_widget.data[6].y = real_values
            
            # Imaginary part trace (index 7)
            current_fig_widget.data[7].x = k_Fr_values
            current_fig_widget.data[7].y = imag_values
        
        return tracked_mode
    
    # Global variables to track current state
    current_fig_widget = None
    current_points_data = None
    current_dataset = None
    
    # Function to handle point clicks and initiate tracking
    def handle_click(trace, points, selector):
        if not points.point_inds or current_fig_widget is None:
            return
            
        pt_idx = points.point_inds[0]
        
        # Get the data for the clicked point
        if trace.customdata and len(trace.customdata) > pt_idx:
            custom_data = trace.customdata[pt_idx]
            point_idx = custom_data['point_idx']
            selected_pt = current_points_data[point_idx]

            # Update the highlight marker positions
            with current_fig_widget.batch_update():
                # Get the indices of the highlight traces
                highlight_real_idx = 4  # Index of the real highlight trace
                highlight_imag_idx = 5  # Index of the imaginary highlight trace
                
                # Update highlight in real part plot (subplot 1,1)
                current_fig_widget.data[highlight_real_idx].x = [selected_pt['k_Fr']]
                current_fig_widget.data[highlight_real_idx].y = [selected_pt['real']]
                
                # Update highlight in imaginary part plot (subplot 1,2)
                current_fig_widget.data[highlight_imag_idx].x = [selected_pt['k_Fr']]
                current_fig_widget.data[highlight_imag_idx].y = [selected_pt['imag']]
            
            # Get all k values from the dataset
            k_values = sorted(list(set(d['k'] for d in current_dataset)))
            
            # Perform mode tracking
            track_mode(selected_pt, k_values, current_points_data, current_fig_widget)
    
    # Update function when dropdown changes
    def on_m_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            m = change['new']
            with out_disp:
                out_disp.clear_output(wait=True)
                nonlocal current_fig_widget, current_points_data, current_dataset
                
                fig, points_data, dataset = create_plot_for_m(m)
                if fig is None:
                    return
                
                current_points_data = points_data
                current_dataset = dataset
                current_fig_widget = go.FigureWidget(fig)
                
                # Set up click callbacks for all data traces
                for i in range(4):  # We have 4 data traces (resolved/unresolved for real/imag)
                    if i < len(current_fig_widget.data):
                        current_fig_widget.data[i].on_click(handle_click)
                
                display(current_fig_widget)
                
                # Clear tracking output
                with out_tracking:
                    out_tracking.clear_output(wait=True)
                    print("Click on a point to track its mode across k values")
    
    # Initialize the plot
    fig, points_data, dataset = create_plot_for_m(m_value)
    if fig is None:
        return VBox([m_dropdown, out_disp, out_tracking])
    
    current_points_data = points_data
    current_dataset = dataset
    current_fig_widget = go.FigureWidget(fig)

    # Set up the initial callbacks for all data traces
    for i in range(4):  # We have 4 data traces
        if i < len(current_fig_widget.data):
            current_fig_widget.data[i].on_click(handle_click)
    
    # Connect the dropdown callback
    m_dropdown.observe(on_m_change, names='value')
    
    # Initial display
    with out_disp:
        display(current_fig_widget)
    
    with out_tracking:
        print("Click on a point to track its mode across k values")
    
    # Return the full widget layout
    return VBox([
        m_dropdown,
        out_disp,
        out_tracking
    ])

In [ ]:
# Create an interactive mode tracking plot
tracking_widget = create_mode_tracking_interactive(
    all_data, 
    m_value=1, 
    omega=0, 
    bv_freq=1.11, 
    eig_min=-0.01, 
    eig_max=0.05, 
    sim_threshold=0.7,
    eig_weight=0.6, 
    vel_weight=0.4, 
    max_gap_jumps=2
)

# Display the widget
tracking_widget

## Critical points calculation

In [ ]:
def find_rc(r, m, bv_freq):
    """
    Find rc where Omega_bar(rc) = ±bv_freq/m
    
    Parameters:
    -----------
    r : numpy.ndarray
        Array of r values
    m : int
        Azimuthal wavenumber
    bv_freq : float
        Brunt-Väisälä frequency
    
    Returns:
    --------
    rc_plus, rc_minus : tuple
        rc values where Omega_bar = +bv_freq/m and -bv_freq/m respectively
    """
    # Avoid division by zero for m=0
    if m == 0:
        return None, None
    
    # Calculate Omega_bar = (1-exp(-r^2))/r^2
    omega_bar = np.where(r > 1e-10, 
                        (1 - np.exp(-r**2)) / r**2,
                        1.0)  # limit as r->0 is 1
    
    target = bv_freq / m
    
    # Find where omega_bar crosses ±target
    # Using numpy's interp to find the crossing points
    r_positive = None
    r_negative = None
    
    try:
        # Find crossing for +bv_freq/m
        positive_crossings = np.where((omega_bar[:-1] - target) * 
                                    (omega_bar[1:] - target) <= 0)[0]
        if len(positive_crossings) > 0:
            idx = positive_crossings[0]
            r_positive = np.interp(target,
                                 [omega_bar[idx], omega_bar[idx+1]],
                                 [r[idx], r[idx+1]])
        
        # Find crossing for -bv_freq/m
        negative_crossings = np.where((omega_bar[:-1] + target) * 
                                    (omega_bar[1:] + target) <= 0)[0]
        if len(negative_crossings) > 0:
            idx = negative_crossings[0]
            r_negative = np.interp(-target,
                                 [omega_bar[idx], omega_bar[idx+1]],
                                 [r[idx], r[idx+1]])
    except:
        pass
    
    return r_positive, r_negative

# Test for different m values
r = np.linspace(0, 5, 1000)  # Radial coordinate from 0 to 5
for m in [1, 2]:
    rc_plus, rc_minus = find_rc(r, m, all_data[0]['bv_freq'])
    print(f"\nFor m={m}:")
    if rc_plus is not None:
        print(f"rc(Omega_bar = +BV/m): {rc_plus:.4f}")
    else:
        print("rc(Omega_bar = +BV/m): Not found")
    if rc_minus is not None:
        print(f"rc(Omega_bar = -BV/m): {rc_minus:.4f}")
    else:
        print("rc(Omega_bar = -BV/m): Not found")
    
    # Plot Omega_bar and the critical points
    omega_bar = np.where(r > 1e-10, 
                        (1 - np.exp(-r**2)) / r**2,
                        1.0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(r, omega_bar, 'b-', label='Ω_bar(r)')
    plt.axhline(all_data[0]['bv_freq']/m, color='r', linestyle='--', 
                label=f'+BV/m = {all_data[0]["bv_freq"]/m:.4f}')
    plt.axhline(-all_data[0]['bv_freq']/m, color='g', linestyle='--', 
                label=f'-BV/m = {-all_data[0]["bv_freq"]/m:.4f}')
    
    if rc_plus is not None:
        plt.plot(rc_plus, all_data[0]['bv_freq']/m, 'ro', label='rc(+)')
    if rc_minus is not None:
        plt.plot(rc_minus, -all_data[0]['bv_freq']/m, 'go', label='rc(-)')
    
    plt.xlabel('r')
    plt.ylabel('Ω_bar')
    plt.title(f'Ω_bar(r) and Critical Points for m={m}')
    plt.grid(True)
    plt.legend()
    plt.show()


## Compare MacOS calculation against Linux results (old)

In [ ]:
# # Find all reference eigenvalue files
# ref_data_dir = os.path.join(data_dir, "retest")
# ref_files = glob.glob(os.path.join(ref_data_dir, 'bsnsq_eig_MK_*.output'))

# # Process reference files
# ref_data = []
# for file in ref_files:
#     data = parse_eigenvalue_file(file)
#     if data:
#         ref_data.append(data)

# print(f"Processed {len(ref_data)} reference files.")

# # Match and compare data
# for data in all_data:
#     # Find matching reference data
#     matching_ref = next((ref for ref in ref_data 
#                         if ref['m'] == data['m'] and ref['k'] == data['k']), None)
    
#     if matching_ref is None:
#         print(f"No matching reference data for m={data['m']}, k={data['k']}")
#         continue
        
#     # Create comparison plot
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
#     # Plot original data
#     colors = np.where(data['is_resolved'], 'red', 'blue')
#     sizes = np.where(data['is_resolved'], 60, 20)
#     ax1.scatter(data['eigenvalues'].real, data['eigenvalues'].imag, 
#                c=colors, s=sizes, alpha=0.7)
#     ax1.set_xlabel('Re(sig)')
#     ax1.set_ylabel('Im(sig)')
#     ax1.set_title('Original Data Eigenvalues')
#     ax1.grid(True, alpha=0.3)
#     ax1.axhline(y=0, color='k', linestyle='-', alpha=0.3)
#     ax1.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    
#     # Plot reference data
#     ref_colors = np.where(matching_ref['is_resolved'], 'red', 'blue')
#     ref_sizes = np.where(matching_ref['is_resolved'], 60, 20)
#     ax2.scatter(matching_ref['eigenvalues'].real, matching_ref['eigenvalues'].imag, 
#                 c=ref_colors, s=ref_sizes, alpha=0.7)
#     ax2.set_xlabel('Re(sig)')
#     ax2.set_ylabel('Im(sig)')
#     ax2.set_title('Reference Data Eigenvalues')
#     ax2.grid(True, alpha=0.3)
#     ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
#     ax2.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    
#     # Create custom legend for resolved/unresolved
#     legend_elements = [
#         Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
#                markersize=10, label='Resolved'),
#         Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
#                markersize=6, label='Unresolved')
#     ]
    
#     # Add legend to both plots
#     ax1.legend(handles=legend_elements, loc='best')
#     ax2.legend(handles=legend_elements, loc='best')
    
#     # Add main title with parameters
#     fig.suptitle(f'Comparison for BV={data["bv_freq"]}, m={data["m"]}, k={data["k"]}, NRCHOP={data["nrchop"]}', 
#                  fontsize=14, y=1.02)
    
#     plt.tight_layout()
#     plt.show()
    
#     # Display statistics
#     stats = pd.DataFrame({
#         'Parameter': ['M Value', 'K Value', 
#                      'Original Total Eigenvalues', 'Reference Total Eigenvalues',
#                      'Original Resolved', 'Reference Resolved'],
#         'Value': [data['m'], data['k'],
#                  len(data['eigenvalues']), len(matching_ref['eigenvalues']),
#                  np.sum(data['is_resolved']), np.sum(matching_ref['is_resolved'])]
#     })
    
#     display(stats)


## Centrifugal instability

In [ ]:
def plot_centrifugal_instability(omega, omega_bar_func, r_range=(0.1, 5), n_points=1000):
    """
    Plot centrifugal instability analysis for given parameters.
    
    Parameters:
    omega: float - Background rotation parameter
    omega_bar_func: callable - Function that takes r and returns omega_bar(r)
    r_range: tuple - (r_min, r_max) for the radial range
    n_points: int - Number of points to evaluate
    
    Returns:
    dict - Analysis results including unstable regions and statistics
    """
    
    # Radial coordinate array
    r_vals = np.linspace(r_range[0], r_range[1], n_points)
    
    # Define symbolic variables for analytical derivatives
    r = sp.Symbol('r', positive=True)
    
    # Convert omega_bar_func to symbolic if it's a lambda function
    # For the specific case of (1 - exp(-r^2))/r^2
    omega_bar_sym = (1 - sp.exp(-r**2)) / r**2
    r_omega_bar_sym = r * omega_bar_sym
    d_r_omega_bar_dr_sym = sp.diff(r_omega_bar_sym, r)
    zeta_sym = omega_bar_sym + d_r_omega_bar_dr_sym
    
    # Calculate the centrifugal instability criterion symbolically
    instability_criterion_sym = (omega_bar_sym + omega) * (2*omega + zeta_sym)
    
    # Convert symbolic expressions to numerical functions
    zeta_func = sp.lambdify(r, zeta_sym, 'numpy')
    instability_criterion_func = sp.lambdify(r, instability_criterion_sym, 'numpy')
    
    # Evaluate at numerical values
    omega_bar = omega_bar_func(r_vals)
    zeta = zeta_func(r_vals)
    instability_criterion = instability_criterion_func(r_vals)
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    
    # Plot the instability criterion
    plt.subplot(1, 2, 1)
    plt.plot(r_vals, instability_criterion, 'b-', linewidth=2, label='(Ω̄+Ω)(2Ω+ζ)')
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.7, label='Stability boundary')
    plt.xlabel('r')
    plt.ylabel('(Ω̄+Ω)(2Ω+ζ)')
    plt.title(f'Centrifugal Instability Criterion (Ω={omega})')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Fill regions where criterion is negative (unstable)
    negative_mask = instability_criterion < 0
    if np.any(negative_mask):
        plt.fill_between(r_vals, instability_criterion, 0, where=negative_mask, 
                         color='red', alpha=0.3, label='Unstable region')
    
    # Individual components with opposite sign regions highlighted
    plt.subplot(1, 2, 2)
    
    # Plot the individual terms
    term1 = omega_bar + omega  # (Ω̄+Ω)
    term2 = 2*omega + zeta  # (2Ω+ζ)
    
    plt.plot(r_vals, term1, 'g-', linewidth=2, label='Ω̄+Ω')
    plt.plot(r_vals, term2, 'purple', linewidth=2, label='2Ω+ζ')
    plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    
    # Highlight regions where the two terms have opposite signs
    opposite_signs = (term1 * term2) < 0
    plt.fill_between(r_vals, plt.ylim()[0], plt.ylim()[1], where=opposite_signs, 
                     color='orange', alpha=0.2, label='Opposite signs')
    
    plt.xlabel('r')
    plt.ylabel('Value')
    plt.title('Individual Terms of Instability Criterion')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.ylim(-1.5, 2)  # Set reasonable y-limits for clarity
    
    plt.tight_layout()
    plt.show()
    
    # Calculate statistics
    results = {
        'omega': omega,
        'r_range': r_range,
        'min_criterion': instability_criterion.min(),
        'max_criterion': instability_criterion.max(),
        'r_vals': r_vals,
        'instability_criterion': instability_criterion,
        'term1': term1,
        'term2': term2
    }
    
    # Find unstable regions
    unstable_regions = instability_criterion < 0
    if np.any(unstable_regions):
        unstable_r = r_vals[unstable_regions]
        results['unstable_region'] = (unstable_r.min(), unstable_r.max())
    else:
        results['unstable_region'] = None
    
    # Find sign changes
    results['term1_zero_crossing'] = r_vals[np.argmin(np.abs(term1))]
    results['term2_zero_crossing'] = r_vals[np.argmin(np.abs(term2))]
    
    # Find regions where terms have opposite signs
    if np.any(opposite_signs):
        opposite_r = r_vals[opposite_signs]
        results['opposite_signs_region'] = (opposite_r.min(), opposite_r.max())
    else:
        results['opposite_signs_region'] = None
    
    # Print statistics
    print(f"Parameters: Ω = {omega}")
    print(f"Range of r: {r_range[0]:.2f} to {r_range[1]:.2f}")
    print(f"Minimum value of (Ω̄+Ω)(2Ω+ζ): {results['min_criterion']:.6f}")
    print(f"Maximum value of (Ω̄+Ω)(2Ω+ζ): {results['max_criterion']:.6f}")
    
    if results['unstable_region']:
        print(f"Unstable region exists from r = {results['unstable_region'][0]:.3f} to r = {results['unstable_region'][1]:.3f}")
    else:
        print("No unstable regions found in the given r range")
    
    print(f"\nSign analysis:")
    print(f"Ω̄+Ω changes sign at r ≈ {results['term1_zero_crossing']:.3f}")
    print(f"2Ω+ζ changes sign at r ≈ {results['term2_zero_crossing']:.3f}")
    
    if results['opposite_signs_region']:
        print(f"Terms have opposite signs from r = {results['opposite_signs_region'][0]:.3f} to r = {results['opposite_signs_region'][1]:.3f}")
    
    return results


In [ ]:
omega_func = lambda r: (1 - np.exp(-r**2)) / r**2
results = plot_centrifugal_instability(omega=-0, omega_bar_func=omega_func)

In [ ]:
def find_unstable_rotation_range(omega_bar_func, r_range=(0.1, 5), n_points=1000, 
                                omega_range=(-2.0, 2.0), n_omega_samples=100, 
                                plot_results=True, tol=1e-6):
    """
    Find the range of background rotation rates (Ω) where centrifugal instability occurs
    for a given angular velocity profile.
    
    Parameters:
    -----------
    omega_bar_func : callable
        Function that takes r and returns omega_bar(r) - the angular velocity profile
    r_range : tuple
        (r_min, r_max) for the radial range to analyze
    n_points : int
        Number of points to evaluate in the radial direction
    omega_range : tuple
        (omega_min, omega_max) range to search for instability
    n_omega_samples : int
        Number of omega values to test
    plot_results : bool
        Whether to create visualization plots
    tol : float
        Tolerance for considering a value negative (for numerical stability)
    
    Returns:
    --------
    dict
        Dictionary containing:
        - unstable_omega_ranges: List of (omega_min, omega_max) tuples representing unstable ranges
        - critical_values: Dictionary with specific critical omega values
        - omega_min_values: Array of minimum criterion values for each omega
        - omega_values: Array of tested omega values
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import sympy as sp
    from matplotlib.colors import LinearSegmentedColormap
    
    # Radial coordinate array
    r_vals = np.linspace(r_range[0], r_range[1], n_points)
    omega_vals = np.linspace(omega_range[0], omega_range[1], n_omega_samples)
    
    # Set up symbolic expressions for calculations
    r = sp.Symbol('r', positive=True)
    
    # For the specific case of the Gaussian vortex: (1 - exp(-r^2))/r^2
    omega_bar_sym = (1 - sp.exp(-r**2)) / r**2
    r_omega_bar_sym = r * omega_bar_sym
    d_r_omega_bar_dr_sym = sp.diff(r_omega_bar_sym, r)
    zeta_sym = omega_bar_sym + d_r_omega_bar_dr_sym
    
    # Convert symbolic expressions to numerical functions
    zeta_func = sp.lambdify(r, zeta_sym, 'numpy')
    
    # Calculate zeta values (vorticity)
    omega_bar = omega_bar_func(r_vals)
    zeta = zeta_func(r_vals)
    
    # Initialize arrays to store results
    min_criterion_values = np.zeros(n_omega_samples)
    criterion_sign_changes = np.zeros(n_omega_samples, dtype=bool)
    unstable_points = np.zeros((n_omega_samples, n_points), dtype=bool)
    
    # Calculate instability for each omega value
    for i, omega in enumerate(omega_vals):
        # Calculate the terms of the instability criterion
        term1 = omega_bar + omega  # (Ω̄+Ω)
        term2 = 2*omega + zeta     # (2Ω+ζ)
        
        # The full criterion
        instability_criterion = term1 * term2
        
        # Store the minimum value
        min_criterion_values[i] = np.min(instability_criterion)
        
        # Check if the criterion changes sign (which means it crosses through negative values)
        # This accounts for cases where we don't sample the most negative point
        criterion_sign_changes[i] = np.any(np.diff(np.signbit(instability_criterion)))
        
        # Mark points where the criterion is negative
        unstable_points[i] = instability_criterion < -tol
    
    # Find ranges of omega where the flow is unstable
    is_unstable = (min_criterion_values < -tol) | criterion_sign_changes
    
    # Identify connected regions of instability
    unstable_ranges = []
    in_unstable_region = False
    region_start = None
    
    for i, unstable in enumerate(is_unstable):
        if unstable and not in_unstable_region:
            # Start of an unstable region
            in_unstable_region = True
            region_start = omega_vals[i]
        elif not unstable and in_unstable_region:
            # End of an unstable region
            in_unstable_region = False
            unstable_ranges.append((region_start, omega_vals[i-1]))
    
    # Check if we're still in an unstable region at the end of the range
    if in_unstable_region:
        unstable_ranges.append((region_start, omega_vals[-1]))
    
    # Find critical values
    critical_values = {}
    
    # Find where term1 changes sign across r for different omega values
    for i, omega in enumerate(omega_vals):
        term1 = omega_bar + omega
        term1_sign_changes = np.where(np.diff(np.signbit(term1)))[0]
        
        if len(term1_sign_changes) > 0:
            r_critical = r_vals[term1_sign_changes[0]]
            critical_values[f'omega={omega:.4f}'] = {
                'term1_zero_r': r_critical,
                'omega_bar_at_zero': -omega
            }
    
    # Create visualization
    if plot_results:
        # Create a colormap to visualize instability
        plt.figure(figsize=(16, 10))
        
        # 1. Plot minimum criterion value vs. omega
        plt.subplot(2, 2, 4)
        plt.text(0.02, 0.98, '(d)', transform=plt.gca().transAxes, fontsize=14, fontweight='bold', verticalalignment='top')
        plt.plot(omega_vals, min_criterion_values, color='black', linestyle='-', linewidth=2)
        plt.axhline(y=0, color='black', linestyle='--', alpha=0.7)
        plt.xlabel(r'$\Omega$', fontsize=14)
        plt.ylabel(r'min$[\phi(r)]$', fontsize=14)
        plt.title('Minimum Value of Instability Criterion')
        plt.grid(True, alpha=0.3)
        
        # Highlight unstable regions
        for start, end in unstable_ranges:
            plt.axvspan(start, end, color='darkgray', alpha=0.7)
        
        # 2. Plot instability criterion as a 2D heatmap
        plt.subplot(2, 2, 3)
        plt.text(0.02, 0.98, '(c)', transform=plt.gca().transAxes, fontsize=14, fontweight='bold', verticalalignment='top')
        # Create a meshgrid for visualization
        R, O = np.meshgrid(r_vals, omega_vals)
        # Calculate instability criterion for each point
        C = np.zeros_like(R)
        for i, omega in enumerate(omega_vals):
            term1 = omega_bar + omega
            term2 = 2*omega + zeta
            C[i, :] = term1 * term2

        # Create a custom colormap: dark gray for instability (negative values), white for stability (positive values)
        cmap = LinearSegmentedColormap.from_list(
            'StabilityMap', [(0, 'darkgray'), (0.5, 'darkgray'), (0.5001, 'white'), (1, 'white')])
        
        # Plot the criterion with a custom colormap
        im = plt.pcolormesh(R, O, C, cmap=cmap, shading='auto', vmin=-0.5, vmax=0.5)
        # plt.colorbar(im, label='Instability Criterion')
        plt.xlabel(r'$r$', fontsize=14)
        plt.ylabel(r'$\Omega$', fontsize=14)
        plt.title('Instability Criterion Map')
        
        # Draw the zero contour to highlight the boundary of stability
        plt.contour(R, O, C, levels=[0], colors='black', linestyles='solid', linewidths=1)
        
        # 3. Plot angular velocity profile
        plt.subplot(2, 2, 1)
        plt.text(0.02, 0.98, '(a)', transform=plt.gca().transAxes, fontsize=14, fontweight='bold', verticalalignment='top')
        plt.plot(r_vals, omega_bar, 'k-', linewidth=2, label=r'$\bar{\Omega}(r)$')
        plt.xlabel(r'$r$', fontsize=14)
        plt.ylabel(r'$\bar{\Omega}$', fontsize=14)
        plt.title(r'$\bar{\Omega}$')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # 4. Plot vorticity profile
        plt.subplot(2, 2, 2)
        plt.text(0.02, 0.98, '(b)', transform=plt.gca().transAxes, fontsize=14, fontweight='bold', verticalalignment='top')
        plt.plot(r_vals, zeta, 'black', linewidth=2, label=r'$\zeta(r)$')
        plt.xlabel(r'$r$', fontsize=14)
        plt.ylabel(r'$\zeta$', fontsize=14)
        plt.title(r'Axial Vorticity Profile $\zeta$')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        plt.tight_layout()
        plt.show()

    # Create and return the results dictionary
    results = {
        'unstable_omega_ranges': unstable_ranges,
        'critical_values': critical_values,
        'omega_min_values': min_criterion_values,
        'omega_values': omega_vals,
        'r_values': r_vals,
        'omega_bar': omega_bar,
        'zeta': zeta
    }
    
    # Print a summary of results
    print("Summary of Centrifugal Instability Analysis:")
    print("-" * 50)
    
    if unstable_ranges:
        print(f"Found {len(unstable_ranges)} unstable rotation rate range(s):")
        for i, (start, end) in enumerate(unstable_ranges):
            print(f"  Range {i+1}: Ω ∈ [{start:.4f}, {end:.4f}]")
    else:
        print("No unstable rotation rate ranges found within the specified Ω range.")
    
    # Calculate percentage of tested omega values that are unstable
    percent_unstable = 100 * np.sum(is_unstable) / len(omega_vals)
    print(f"Percentage of tested Ω values that produce instability: {percent_unstable:.1f}%")
    
    return results

In [ ]:
# Gaussian vortex profile
omega_func = lambda r: (1 - np.exp(-r**2)) / r**2

# Find unstable rotation ranges
results = find_unstable_rotation_range(
    omega_bar_func=omega_func,
    r_range=(0.1, 5),
    omega_range=(-1.5, 1.5),
    n_omega_samples=200,
    plot_results=True
)